In [1]:
# ==================================================================================================
# CISLR — L28 FULL STANDALONE
# REAL-TIME TEMPORAL STABILIZATION
# ==================================================================================================
#
# FRESH-KERNEL COMPATIBLE
#
# Complete pipeline:
#
# Webcam
#   ↓
# OpenVINO Palm Detector
#   ↓
# Weighted NMS
#   ↓
# Rotated Hand ROI
#   ↓
# OpenVINO 21-point Hand Landmarks
#   ↓
# L26C 63-D Landmark Representation
#   ↓
# Rolling 32-frame Sequence
#   ↓
# OpenVINO BiGRU
#   ↓
# L27 NO_SIGN Rejection
#   ↓
# L28 Temporal Stabilization
#   ↓
# STABLE SIGN / NO_SIGN
#
# NO VideoMAE
# NO retraining
# NO locked test
# NO sentence generation yet
#
# Controls:
#   Q / ESC = quit
#   R       = reset temporal state
#
# ==================================================================================================

from pathlib import Path
from collections import deque, Counter
import cv2
import json
import math
import time
import hashlib
import numpy as np
import openvino as ov


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


PALM_MODEL_PATH = (
    LANDMARK_ROOT /
    "models/pretrained/palm/"
    "palm_detection_full.tflite"
)


HAND_MODEL_PATH = (
    LANDMARK_ROOT /
    "models/pretrained/hand_landmark/"
    "hand_landmark_full.tflite"
)


CLASSIFIER_PATH = (
    LANDMARK_ROOT /
    "audit/l26c_realtime_landmark_classifier/"
    "models/l26c_fast_landmark_bigru.xml"
)


L27_THRESHOLD_PATH = (
    LANDMARK_ROOT /
    "audit/l27_nosign_rejection/"
    "config/l27_rejection_thresholds.json"
)


L28_ROOT = (
    LANDMARK_ROOT /
    "audit/l28_temporal_stabilization"
)


REPORT_ROOT = (
    L28_ROOT /
    "reports"
)


CONFIG_ROOT = (
    L28_ROOT /
    "config"
)


REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 1. BASIC CONFIG
# ==================================================================================================

CAMERA_ID = 0

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480


PALM_SIZE = 192

LANDMARK_SIZE = 224


PALM_THRESHOLD = 0.50

NMS_THRESHOLD = 0.30

MAX_HANDS = 2


SEQUENCE_LENGTH = 32

FEATURE_DIM = 63


CLASS_NAMES = [
    "monday",
    "interest",
    "thursday",
    "christmas",
    "pencil",
    "mother",
    "tuesday",
    "friday",
    "rubber",
    "wash"
]


HAND_CONNECTIONS = [

    (0,1),
    (1,2),
    (2,3),
    (3,4),

    (0,5),
    (5,6),
    (6,7),
    (7,8),

    (5,9),
    (9,10),
    (10,11),
    (11,12),

    (9,13),
    (13,14),
    (14,15),
    (15,16),

    (13,17),
    (17,18),
    (18,19),
    (19,20),

    (0,17)
]


# ==================================================================================================
# 2. L28 PARAMETERS
# ==================================================================================================
#
# These are L28 development parameters.
#
# Window = recent L27 outputs.
#
# Initial sign:
#   >= 4 matching votes in last 7
#   AND >= 55% of history
#
# Sign switch:
#   stronger evidence = 5 votes
#
# Return to NO_SIGN:
#   4 consecutive rejected windows
#
# ==================================================================================================

STABILITY_WINDOW = 7

REQUIRED_VOTES = 4

REQUIRED_RATIO = 0.55

CHANGE_VOTES = 5

RELEASE_NO_SIGN = 4


# ==================================================================================================
# 3. VERIFY FILES
# ==================================================================================================

print("=" * 120)
print("CISLR — L28 FULL STANDALONE INITIALIZATION")
print("=" * 120)


required_files = {

    "Palm detector":
        PALM_MODEL_PATH,

    "Hand landmarks":
        HAND_MODEL_PATH,

    "L26C BiGRU":
        CLASSIFIER_PATH,

    "L27 thresholds":
        L27_THRESHOLD_PATH
}


for name,path in required_files.items():

    print(
        f"{name:<20}: {path}"
    )


    if not path.exists():

        raise FileNotFoundError(
            f"\n{name} not found:\n{path}"
        )


print()
print(
    "[PASS] All required files found."
)


# ==================================================================================================
# 4. LOAD L27 THRESHOLDS
# ==================================================================================================

with open(
    L27_THRESHOLD_PATH
) as f:

    l27_config = json.load(
        f
    )


thresholds = (
    l27_config[
        "thresholds"
    ]
)


CONF_T = float(
    thresholds[
        "confidence_threshold"
    ]
)


MARGIN_T = float(
    thresholds[
        "margin_threshold"
    ]
)


ENTROPY_T = float(
    thresholds[
        "entropy_threshold"
    ]
)


HAND_T = float(
    thresholds[
        "hand_ratio_threshold"
    ]
)


print()
print("=" * 120)
print("L27 THRESHOLDS")
print("=" * 120)

print()

print(
    f"Confidence >= {CONF_T:.3f}"
)

print(
    f"Margin     >= {MARGIN_T:.3f}"
)

print(
    f"Entropy    <= {ENTROPY_T:.3f}"
)

print(
    f"Hand ratio >= {HAND_T:.3f}"
)


# ==================================================================================================
# 5. OPENVINO CORE
# ==================================================================================================

core = ov.Core()


print()
print(
    "OpenVINO:",
    ov.__version__
)

print(
    "Devices :",
    core.available_devices
)


# ==================================================================================================
# 6. LOAD PALM
# ==================================================================================================

print()
print(
    "Loading palm detector..."
)


palm_model = core.read_model(
    str(
        PALM_MODEL_PATH
    )
)


palm_compiled = core.compile_model(
    palm_model,
    "CPU",
    {
        "PERFORMANCE_HINT":
            "LATENCY"
    }
)


palm_input = (
    palm_compiled.input(0)
)


palm_outputs = [

    palm_compiled.output(i)

    for i in range(
        len(
            palm_compiled.outputs
        )
    )
]


print(
    "[PASS] Palm:",
    palm_input.partial_shape
)


# ==================================================================================================
# 7. LOAD LANDMARK
# ==================================================================================================

print(
    "Loading landmark detector..."
)


hand_model = core.read_model(
    str(
        HAND_MODEL_PATH
    )
)


hand_compiled = core.compile_model(
    hand_model,
    "CPU",
    {
        "PERFORMANCE_HINT":
            "LATENCY"
    }
)


hand_input = (
    hand_compiled.input(0)
)


hand_outputs = [

    hand_compiled.output(i)

    for i in range(
        len(
            hand_compiled.outputs
        )
    )
]


print(
    "[PASS] Landmark:",
    hand_input.partial_shape
)


# ==================================================================================================
# 8. LOAD CLASSIFIER
# ==================================================================================================

print(
    "Loading temporal classifier..."
)


classifier_model = (
    core.read_model(
        str(
            CLASSIFIER_PATH
        )
    )
)


classifier_compiled = (
    core.compile_model(
        classifier_model,
        "CPU",
        {
            "PERFORMANCE_HINT":
                "LATENCY"
        }
    )
)


classifier_input = (
    classifier_compiled.input(0)
)


classifier_output = (
    classifier_compiled.output(0)
)


print(
    "[PASS] BiGRU input :",
    classifier_input.partial_shape
)

print(
    "[PASS] BiGRU output:",
    classifier_output.partial_shape
)


# ==================================================================================================
# 9. MODEL LAYOUT
# ==================================================================================================

def get_shape(
    port
):

    try:

        return tuple(
            int(x)
            for x in port.shape
        )

    except Exception:

        return None


def is_nhwc(
    shape
):

    if shape is None:
        return True


    return (

        len(shape) == 4

        and

        shape[-1] == 3
    )


PALM_NHWC = is_nhwc(
    get_shape(
        palm_input
    )
)


HAND_NHWC = is_nhwc(
    get_shape(
        hand_input
    )
)


print()
print(
    "Palm layout:",
    "NHWC"
    if PALM_NHWC
    else "NCHW"
)

print(
    "Hand layout:",
    "NHWC"
    if HAND_NHWC
    else "NCHW"
)


# ==================================================================================================
# 10. PALM ANCHORS
# ==================================================================================================

def generate_palm_anchors():

    anchors = []


    strides = [
        8,
        16,
        16,
        16
    ]


    layer = 0


    while layer < len(
        strides
    ):

        stride = strides[
            layer
        ]


        last = layer


        while (

            last + 1
            <
            len(strides)

            and

            strides[
                last + 1
            ]
            ==
            stride
        ):

            last += 1


        layers_same_stride = (
            last -
            layer +
            1
        )


        anchors_per_cell = (
            2 *
            layers_same_stride
        )


        feature_size = int(
            math.ceil(
                PALM_SIZE /
                stride
            )
        )


        for y in range(
            feature_size
        ):

            for x in range(
                feature_size
            ):

                cx = (
                    x + 0.5
                ) / feature_size


                cy = (
                    y + 0.5
                ) / feature_size


                for _ in range(
                    anchors_per_cell
                ):

                    anchors.append(
                        [
                            cx,
                            cy,
                            1.0,
                            1.0
                        ]
                    )


        layer = (
            last + 1
        )


    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )


    if anchors.shape != (
        2016,
        4
    ):

        raise RuntimeError(
            "Anchor generation failed: "
            f"{anchors.shape}"
        )


    return anchors


PALM_ANCHORS = (
    generate_palm_anchors()
)


print()
print(
    "[PASS] Palm anchors:",
    PALM_ANCHORS.shape
)


# ==================================================================================================
# 11. SIGMOID
# ==================================================================================================

def sigmoid(
    x
):

    x = np.clip(
        x,
        -80,
        80
    )


    return (
        1.0 /
        (
            1.0 +
            np.exp(
                -x
            )
        )
    )


# ==================================================================================================
# 12. LETTERBOX
# ==================================================================================================

def letterbox(
    frame,
    target
):

    h,w = frame.shape[:2]


    scale = min(
        target / w,
        target / h
    )


    new_w = int(
        round(
            w * scale
        )
    )


    new_h = int(
        round(
            h * scale
        )
    )


    resized = cv2.resize(
        frame,
        (
            new_w,
            new_h
        )
    )


    canvas = np.zeros(
        (
            target,
            target,
            3
        ),
        dtype=np.uint8
    )


    left = (
        target -
        new_w
    ) // 2


    top = (
        target -
        new_h
    ) // 2


    canvas[
        top:top+new_h,
        left:left+new_w
    ] = resized


    return (
        canvas,
        scale,
        left,
        top
    )


# ==================================================================================================
# 13. MODEL INPUT
# ==================================================================================================

def prepare_input(
    rgb,
    nhwc
):

    x = (
        rgb.astype(
            np.float32
        )
        /
        255.0
    )


    if nhwc:

        x = x[
            None,
            ...
        ]


    else:

        x = np.transpose(
            x,
            (
                2,
                0,
                1
            )
        )


        x = x[
            None,
            ...
        ]


    return x.astype(
        np.float32
    )


# ==================================================================================================
# 14. PALM OUTPUTS
# ==================================================================================================

def get_palm_outputs(
    result
):

    regression = None

    scores = None


    for output in palm_outputs:

        arr = np.asarray(
            result[
                output
            ]
        )


        if arr.size == (
            2016 * 18
        ):

            regression = (
                arr.reshape(
                    2016,
                    18
                )
            )


        elif arr.size == 2016:

            scores = (
                arr.reshape(
                    2016
                )
            )


    if regression is None:

        raise RuntimeError(
            "Palm regression output not found."
        )


    if scores is None:

        raise RuntimeError(
            "Palm score output not found."
        )


    return (
        regression.astype(
            np.float32
        ),
        scores.astype(
            np.float32
        )
    )


# ==================================================================================================
# 15. PALM DECODER
# ==================================================================================================

def decode_palms(
    regression,
    logits
):

    probs = sigmoid(
        logits
    )


    indices = np.where(
        probs >= PALM_THRESHOLD
    )[0]


    detections = []


    for idx in indices:

        raw = regression[
            idx
        ]


        anchor = PALM_ANCHORS[
            idx
        ]


        cx = (
            raw[0] /
            PALM_SIZE
            +
            anchor[0]
        )


        cy = (
            raw[1] /
            PALM_SIZE
            +
            anchor[1]
        )


        bw = (
            raw[2] /
            PALM_SIZE
        )


        bh = (
            raw[3] /
            PALM_SIZE
        )


        box = np.asarray(
            [
                cx - bw/2,
                cy - bh/2,
                cx + bw/2,
                cy + bh/2
            ],
            dtype=np.float32
        )


        keypoints = []


        for k in range(7):

            kx = (
                raw[
                    4 + 2*k
                ]
                /
                PALM_SIZE
                +
                anchor[0]
            )


            ky = (
                raw[
                    5 + 2*k
                ]
                /
                PALM_SIZE
                +
                anchor[1]
            )


            keypoints.append(
                [
                    kx,
                    ky
                ]
            )


        detections.append(
            {
                "score":
                    float(
                        probs[
                            idx
                        ]
                    ),

                "box":
                    box,

                "keypoints":
                    np.asarray(
                        keypoints,
                        dtype=np.float32
                    )
            }
        )


    return detections


# ==================================================================================================
# 16. IOU
# ==================================================================================================

def box_iou(
    a,
    b
):

    x1 = max(
        float(a[0]),
        float(b[0])
    )


    y1 = max(
        float(a[1]),
        float(b[1])
    )


    x2 = min(
        float(a[2]),
        float(b[2])
    )


    y2 = min(
        float(a[3]),
        float(b[3])
    )


    iw = max(
        0.0,
        x2 - x1
    )


    ih = max(
        0.0,
        y2 - y1
    )


    intersection = (
        iw * ih
    )


    area_a = (

        max(
            0.0,
            float(
                a[2] -
                a[0]
            )
        )

        *

        max(
            0.0,
            float(
                a[3] -
                a[1]
            )
        )
    )


    area_b = (

        max(
            0.0,
            float(
                b[2] -
                b[0]
            )
        )

        *

        max(
            0.0,
            float(
                b[3] -
                b[1]
            )
        )
    )


    union = (
        area_a +
        area_b -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection /
        union
    )


# ==================================================================================================
# 17. WEIGHTED NMS
# ==================================================================================================

def weighted_nms(
    detections
):

    if not detections:

        return []


    remaining = sorted(
        detections,
        key=lambda d:
            d["score"],
        reverse=True
    )


    output = []


    while remaining:

        reference = (
            remaining[0]
        )


        overlap = []

        rest = []


        for d in remaining:

            if (
                box_iou(
                    reference["box"],
                    d["box"]
                )
                >
                NMS_THRESHOLD
            ):

                overlap.append(
                    d
                )

            else:

                rest.append(
                    d
                )


        if len(overlap) == 1:

            merged = (
                overlap[0]
            )


        else:

            weights = np.asarray(
                [
                    d["score"]
                    for d in overlap
                ],
                dtype=np.float32
            )


            weights /= (
                np.sum(
                    weights
                )
            )


            merged_box = np.sum(

                np.stack(
                    [
                        d["box"]
                        for d in overlap
                    ]
                )

                *

                weights[:,None],

                axis=0
            )


            merged_kp = np.sum(

                np.stack(
                    [
                        d["keypoints"]
                        for d in overlap
                    ]
                )

                *

                weights[
                    :,
                    None,
                    None
                ],

                axis=0
            )


            merged = {

                "score":
                    float(
                        max(
                            d["score"]
                            for d in overlap
                        )
                    ),

                "box":
                    merged_box.astype(
                        np.float32
                    ),

                "keypoints":
                    merged_kp.astype(
                        np.float32
                    )
            }


        output.append(
            merged
        )


        remaining = rest


    return output[
        :MAX_HANDS
    ]


# ==================================================================================================
# 18. REMOVE LETTERBOX
# ==================================================================================================

def to_original_coordinates(
    detection,
    scale,
    left,
    top
):

    box = (
        detection[
            "box"
        ].copy()
    )


    kp = (
        detection[
            "keypoints"
        ].copy()
    )


    box[
        [0,2]
    ] = (

        box[
            [0,2]
        ]
        *
        PALM_SIZE

        -
        left

    ) / scale


    box[
        [1,3]
    ] = (

        box[
            [1,3]
        ]
        *
        PALM_SIZE

        -
        top

    ) / scale


    kp[:,0] = (

        kp[:,0]
        *
        PALM_SIZE

        -
        left

    ) / scale


    kp[:,1] = (

        kp[:,1]
        *
        PALM_SIZE

        -
        top

    ) / scale


    return {

        "score":
            detection[
                "score"
            ],

        "box":
            box,

        "keypoints":
            kp
    }


# ==================================================================================================
# 19. ANGLE
# ==================================================================================================

def normalize_angle(
    angle
):

    return (

        angle

        -

        2.0 *
        math.pi

        *

        math.floor(

            (
                angle +
                math.pi
            )

            /

            (
                2.0 *
                math.pi
            )
        )
    )


# ==================================================================================================
# 20. PALM -> ROI
# ==================================================================================================

def palm_to_roi(
    detection
):

    box = detection[
        "box"
    ]


    kp = detection[
        "keypoints"
    ]


    wrist = kp[0]

    middle = kp[2]


    rotation = normalize_angle(

        math.pi / 2.0

        -

        math.atan2(

            -(
                middle[1]
                -
                wrist[1]
            ),

            (
                middle[0]
                -
                wrist[0]
            )
        )
    )


    rect_w = max(
        1.0,
        float(
            box[2] -
            box[0]
        )
    )


    rect_h = max(
        1.0,
        float(
            box[3] -
            box[1]
        )
    )


    cx = (
        float(
            box[0]
        )
        +
        float(
            box[2]
        )
    ) / 2.0


    cy = (
        float(
            box[1]
        )
        +
        float(
            box[3]
        )
    ) / 2.0


    shift_y = -0.5


    x_shift = (

        -
        rect_h
        *
        shift_y
        *
        math.sin(
            rotation
        )
    )


    y_shift = (

        rect_h
        *
        shift_y
        *
        math.cos(
            rotation
        )
    )


    cx += x_shift

    cy += y_shift


    rect_w *= 2.6

    rect_h *= 2.6


    size = max(
        rect_w,
        rect_h
    )


    return {

        "cx":
            cx,

        "cy":
            cy,

        "width":
            size,

        "height":
            size,

        "rotation":
            rotation
    }


# ==================================================================================================
# 21. ROI CORNERS
# ==================================================================================================

def roi_corners(
    roi
):

    w = roi[
        "width"
    ]


    h = roi[
        "height"
    ]


    local = np.asarray(
        [
            [-w/2,-h/2],
            [ w/2,-h/2],
            [ w/2, h/2],
            [-w/2, h/2]
        ],
        dtype=np.float32
    )


    angle = roi[
        "rotation"
    ]


    c = math.cos(
        angle
    )


    s = math.sin(
        angle
    )


    R = np.asarray(
        [
            [c,-s],
            [s,c]
        ],
        dtype=np.float32
    )


    points = (
        local @ R.T
    )


    points[:,0] += (
        roi["cx"]
    )


    points[:,1] += (
        roi["cy"]
    )


    return points.astype(
        np.float32
    )


# ==================================================================================================
# 22. WARP HAND
# ==================================================================================================

def warp_hand(
    frame,
    roi
):

    corners = roi_corners(
        roi
    )


    src = np.asarray(
        [
            corners[0],
            corners[1],
            corners[3]
        ],
        dtype=np.float32
    )


    dst = np.asarray(
        [
            [0,0],

            [
                LANDMARK_SIZE - 1,
                0
            ],

            [
                0,
                LANDMARK_SIZE - 1
            ]
        ],
        dtype=np.float32
    )


    M = cv2.getAffineTransform(
        src,
        dst
    )


    crop = cv2.warpAffine(
        frame,
        M,
        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT
    )


    M_inv = (
        cv2.invertAffineTransform(
            M
        )
    )


    return (
        crop,
        M_inv,
        corners
    )


# ==================================================================================================
# 23. LANDMARK OUTPUT
# ==================================================================================================

def get_screen_landmarks(
    result
):

    candidates = []


    for output in hand_outputs:

        arr = np.asarray(
            result[
                output
            ]
        )


        if arr.size == 63:

            candidates.append(

                arr.reshape(
                    21,
                    3
                ).astype(
                    np.float32
                )
            )


    if not candidates:

        raise RuntimeError(
            "63-value landmark output not found."
        )


    # Select screen-space landmarks rather than world landmarks.

    return max(

        candidates,

        key=lambda arr:
            float(
                np.max(
                    np.abs(
                        arr[:,:2]
                    )
                )
            )
    )


# ==================================================================================================
# 24. LANDMARK INVERSE TRANSFORM
# ==================================================================================================

def inverse_landmarks(
    raw,
    M_inv,
    roi
):

    xy = raw[
        :,
        :2
    ].astype(
        np.float32
    )


    ones = np.ones(
        (
            21,
            1
        ),
        dtype=np.float32
    )


    homogeneous = np.concatenate(
        [
            xy,
            ones
        ],
        axis=1
    )


    projected = (
        homogeneous
        @
        M_inv.T
    )


    z = (

        raw[:,2]

        /
        LANDMARK_SIZE

        *
        roi[
            "width"
        ]
    )


    return np.concatenate(
        [
            projected,
            z[:,None]
        ],
        axis=1
    ).astype(
        np.float32
    )


# ==================================================================================================
# 25. COMPLETE HAND DETECTOR
# ==================================================================================================

def detect_hands(
    frame
):

    # ==============================================================================================
    # PALM
    # ==============================================================================================

    t0 = time.perf_counter()


    lb,scale,left,top = (
        letterbox(
            frame,
            PALM_SIZE
        )
    )


    rgb = cv2.cvtColor(
        lb,
        cv2.COLOR_BGR2RGB
    )


    tensor = prepare_input(
        rgb,
        PALM_NHWC
    )


    palm_result = palm_compiled(
        {
            palm_input:
                tensor
        }
    )


    regression,scores = (
        get_palm_outputs(
            palm_result
        )
    )


    detections = decode_palms(
        regression,
        scores
    )


    detections = weighted_nms(
        detections
    )


    detections = [

        to_original_coordinates(
            d,
            scale,
            left,
            top
        )

        for d in detections
    ]


    palm_ms = (
        time.perf_counter()
        -
        t0
    ) * 1000.0


    # ==============================================================================================
    # LANDMARK
    # ==============================================================================================

    t1 = time.perf_counter()


    results = []


    for detection in detections:

        try:

            roi = palm_to_roi(
                detection
            )


            crop,M_inv,corners = (
                warp_hand(
                    frame,
                    roi
                )
            )


            crop_rgb = cv2.cvtColor(
                crop,
                cv2.COLOR_BGR2RGB
            )


            hand_tensor = prepare_input(
                crop_rgb,
                HAND_NHWC
            )


            result = hand_compiled(
                {
                    hand_input:
                        hand_tensor
                }
            )


            raw = get_screen_landmarks(
                result
            )


            landmarks = inverse_landmarks(
                raw,
                M_inv,
                roi
            )


            results.append(
                {
                    "landmarks":
                        landmarks,

                    "roi_corners":
                        corners,

                    "palm_score":
                        detection[
                            "score"
                        ]
                }
            )


        except Exception:

            continue


    landmark_ms = (
        time.perf_counter()
        -
        t1
    ) * 1000.0


    return (
        results,
        palm_ms,
        landmark_ms
    )


# ==================================================================================================
# 26. DRAW SKELETON
# ==================================================================================================

def draw_hand(
    frame,
    result
):

    landmarks = np.asarray(
        result[
            "landmarks"
        ]
    )


    points = landmarks[
        :,
        :2
    ].astype(
        np.int32
    )


    corners = result.get(
        "roi_corners"
    )


    if corners is not None:

        cv2.polylines(
            frame,
            [
                np.asarray(
                    corners,
                    dtype=np.int32
                )
            ],
            True,
            (
                255,
                255,
                0
            ),
            1,
            cv2.LINE_AA
        )


    for a,b in HAND_CONNECTIONS:

        cv2.line(
            frame,
            tuple(
                points[a]
            ),
            tuple(
                points[b]
            ),
            (
                0,
                255,
                0
            ),
            2,
            cv2.LINE_AA
        )


    for i,p in enumerate(
        points
    ):

        radius = (
            5
            if i == 0
            else 3
        )


        cv2.circle(
            frame,
            tuple(p),
            radius,
            (
                0,
                0,
                255
            ),
            -1,
            cv2.LINE_AA
        )


# ==================================================================================================
# 27. CLASSIFIER FEATURE
# ==================================================================================================

def normalize_for_classifier(
    landmarks
):

    xyz = np.asarray(
        landmarks,
        dtype=np.float32
    ).copy()


    if xyz.shape != (
        21,
        3
    ):

        return None


    # Wrist centered.

    xyz -= xyz[
        0:1
    ]


    # Wrist -> middle MCP scale.

    scale = np.linalg.norm(
        xyz[9]
    )


    if (
        not np.isfinite(
            scale
        )
        or
        scale < 1e-6
    ):

        scale = np.max(

            np.linalg.norm(
                xyz,
                axis=1
            )
        )


    if (
        not np.isfinite(
            scale
        )
        or
        scale < 1e-6
    ):

        scale = 1.0


    xyz /= scale


    feature = xyz.reshape(
        FEATURE_DIM
    ).astype(
        np.float32
    )


    if not np.all(
        np.isfinite(
            feature
        )
    ):

        return None


    return feature


# ==================================================================================================
# 28. SOFTMAX
# ==================================================================================================

def softmax(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float32
    )


    logits -= (
        np.max(
            logits
        )
    )


    e = np.exp(
        logits
    )


    return (
        e /
        np.sum(e)
    )


# ==================================================================================================
# 29. CLASSIFY TEMPORAL WINDOW
# ==================================================================================================

def classify_sequence(
    sequence
):

    sequence = np.asarray(
        sequence,
        dtype=np.float32
    )


    if sequence.shape != (
        SEQUENCE_LENGTH,
        FEATURE_DIM
    ):

        raise ValueError(
            f"Expected (32,63), "
            f"got {sequence.shape}"
        )


    x = sequence[
        None,
        :,
        :
    ]


    start = time.perf_counter()


    result = classifier_compiled(
        {
            classifier_input:
                x
        }
    )


    inference_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    logits = np.asarray(
        result[
            classifier_output
        ]
    ).reshape(
        -1
    )


    probabilities = softmax(
        logits
    )


    ranking = np.argsort(
        probabilities
    )[::-1]


    top1_id = int(
        ranking[0]
    )


    top2_id = int(
        ranking[1]
    )


    confidence = float(
        probabilities[
            top1_id
        ]
    )


    top2 = float(
        probabilities[
            top2_id
        ]
    )


    margin = (
        confidence -
        top2
    )


    entropy = float(

        -

        np.sum(
            probabilities
            *
            np.log(
                probabilities +
                1e-12
            )
        )

        /

        np.log(
            len(
                probabilities
            )
        )
    )


    return {

        "prediction":
            CLASS_NAMES[
                top1_id
            ],

        "confidence":
            confidence,

        "margin":
            margin,

        "entropy":
            entropy,

        "inference_ms":
            inference_ms
    }


# ==================================================================================================
# 30. L27 REJECTION FUNCTION
# ==================================================================================================

def l27_reject(
    metrics,
    hand_ratio
):

    accepted = (

        metrics[
            "confidence"
        ]
        >=
        CONF_T

        and

        metrics[
            "margin"
        ]
        >=
        MARGIN_T

        and

        metrics[
            "entropy"
        ]
        <=
        ENTROPY_T

        and

        hand_ratio
        >=
        HAND_T
    )


    if accepted:

        return (
            metrics[
                "prediction"
            ]
        )


    return "NO_SIGN"


# ==================================================================================================
# 31. L28 STABILIZER CLASS
# ==================================================================================================

class TemporalStabilizer:

    def __init__(
        self,
        window_size=7,
        required_votes=4,
        required_ratio=0.55,
        change_votes=5,
        release_nosign=4
    ):

        self.window_size = (
            window_size
        )


        self.required_votes = (
            required_votes
        )


        self.required_ratio = (
            required_ratio
        )


        self.change_votes = (
            change_votes
        )


        self.release_nosign = (
            release_nosign
        )


        self.history = deque(
            maxlen=window_size
        )


        self.stable = (
            "NO_SIGN"
        )


        self.consecutive_nosign = 0


    def reset(
        self
    ):

        self.history.clear()


        self.stable = (
            "NO_SIGN"
        )


        self.consecutive_nosign = 0


    def update(
        self,
        prediction
    ):

        previous_stable = (
            self.stable
        )


        self.history.append(
            prediction
        )


        # ------------------------------------------------------------------------------------------
        # NO_SIGN evidence
        # ------------------------------------------------------------------------------------------

        if prediction == "NO_SIGN":

            self.consecutive_nosign += 1

        else:

            self.consecutive_nosign = 0


        # ------------------------------------------------------------------------------------------
        # RELEASE
        # ------------------------------------------------------------------------------------------

        if (
            self.consecutive_nosign
            >=
            self.release_nosign
        ):

            self.stable = (
                "NO_SIGN"
            )


            return (
                self.stable,
                previous_stable
                !=
                self.stable
            )


        # ------------------------------------------------------------------------------------------
        # Accepted history
        # ------------------------------------------------------------------------------------------

        accepted_history = [

            x

            for x in self.history

            if x != "NO_SIGN"
        ]


        if not accepted_history:

            return (
                self.stable,
                False
            )


        counts = Counter(
            accepted_history
        )


        candidate,votes = (
            counts.most_common(
                1
            )[0]
        )


        # IMPORTANT:
        # denominator is complete recent decision history,
        # including NO_SIGN decisions.

        ratio = (

            votes /
            len(
                self.history
            )
        )


        # ------------------------------------------------------------------------------------------
        # INITIAL ACQUISITION
        # ------------------------------------------------------------------------------------------

        if self.stable == "NO_SIGN":

            if (

                votes
                >=
                self.required_votes

                and

                ratio
                >=
                self.required_ratio
            ):

                self.stable = (
                    candidate
                )


        # ------------------------------------------------------------------------------------------
        # SAME LABEL
        # ------------------------------------------------------------------------------------------

        elif candidate == self.stable:

            pass


        # ------------------------------------------------------------------------------------------
        # SIGN -> DIFFERENT SIGN
        # ------------------------------------------------------------------------------------------

        elif (

            votes
            >=
            self.change_votes

            and

            ratio
            >=
            self.required_ratio
        ):

            self.stable = (
                candidate
            )


        changed = (

            previous_stable
            !=
            self.stable
        )


        return (
            self.stable,
            changed
        )


# ==================================================================================================
# 32. INITIALIZE L28
# ==================================================================================================

stabilizer = TemporalStabilizer(

    window_size=
        STABILITY_WINDOW,

    required_votes=
        REQUIRED_VOTES,

    required_ratio=
        REQUIRED_RATIO,

    change_votes=
        CHANGE_VOTES,

    release_nosign=
        RELEASE_NO_SIGN
)


print()
print("=" * 120)
print("L28 TEMPORAL PARAMETERS")
print("=" * 120)

print()

print(
    "History window     :",
    STABILITY_WINDOW
)

print(
    "Initial votes      :",
    REQUIRED_VOTES
)

print(
    "Required ratio     :",
    REQUIRED_RATIO
)

print(
    "Sign-change votes  :",
    CHANGE_VOTES
)

print(
    "NO_SIGN release    :",
    RELEASE_NO_SIGN
)


# ==================================================================================================
# 33. CAMERA
# ==================================================================================================

cap = cv2.VideoCapture(
    CAMERA_ID
)


if not cap.isOpened():

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    CAMERA_WIDTH
)


cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    CAMERA_HEIGHT
)


try:

    cap.set(
        cv2.CAP_PROP_BUFFERSIZE,
        1
    )

except Exception:

    pass


print()
print(
    "[PASS] Webcam opened."
)


print(
    "Resolution:",
    int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    ),
    "x",
    int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )
)


print()
print(
    "Q / ESC = quit"
)

print(
    "R       = reset temporal state"
)


# ==================================================================================================
# 34. TEMPORAL BUFFERS
# ==================================================================================================

feature_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


hand_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


last_feature = None


# ==================================================================================================
# 35. DISPLAY STATE
# ==================================================================================================

raw_prediction = "-"

l27_prediction = (
    "NO_SIGN"
)

stable_prediction = (
    "NO_SIGN"
)


confidence = 0.0

margin = 0.0

entropy = 1.0


classifier_ms = 0.0


# ==================================================================================================
# 36. SESSION METRICS
# ==================================================================================================

frame_count = 0

hand_frames = 0

classification_windows = 0


l27_accepted = 0

l27_rejected = 0


stable_changes = 0


palm_times = []

landmark_times = []

classifier_times = []

frame_times = []


stable_events = []


fps_frames = 0

fps_start = time.perf_counter()

live_fps = 0.0


session_start = (
    time.perf_counter()
)


print()
print("=" * 120)
print("L28 LIVE SESSION STARTED")
print("=" * 120)


# ==================================================================================================
# 37. LIVE LOOP
# ==================================================================================================

try:

    while True:

        frame_start = (
            time.perf_counter()
        )


        ok,frame = cap.read()


        if not ok:

            continue


        frame_count += 1


        # ==========================================================================================
        # HAND DETECTION
        # ==========================================================================================

        try:

            results,palm_ms,landmark_ms = (
                detect_hands(
                    frame
                )
            )


        except Exception as e:

            results = []

            palm_ms = 0.0

            landmark_ms = 0.0


        palm_times.append(
            float(
                palm_ms
            )
        )


        landmark_times.append(
            float(
                landmark_ms
            )
        )


        hand_present = False

        current_feature = None


        if results:

            current_feature = (
                normalize_for_classifier(
                    results[0][
                        "landmarks"
                    ]
                )
            )


            if current_feature is not None:

                hand_present = True

                hand_frames += 1


                last_feature = (
                    current_feature.copy()
                )


        # ==========================================================================================
        # HAND EVIDENCE
        # ==========================================================================================

        hand_buffer.append(
            int(
                hand_present
            )
        )


        hand_ratio = (

            float(
                np.mean(
                    hand_buffer
                )
            )

            if len(
                hand_buffer
            )

            else 0.0
        )


        # ==========================================================================================
        # FEATURE BUFFER
        # ==========================================================================================

        if current_feature is not None:

            feature_buffer.append(
                current_feature.copy()
            )


        elif last_feature is not None:

            # Brief detector miss:
            # retain previous valid feature.

            feature_buffer.append(
                last_feature.copy()
            )


        # ==========================================================================================
        # CLASSIFICATION
        # ==========================================================================================

        l27_prediction = (
            "NO_SIGN"
        )


        if (
            len(
                feature_buffer
            )
            ==
            SEQUENCE_LENGTH
        ):

            metrics = (
                classify_sequence(
                    np.stack(
                        feature_buffer
                    )
                )
            )


            classification_windows += 1


            raw_prediction = (
                metrics[
                    "prediction"
                ]
            )


            confidence = float(
                metrics[
                    "confidence"
                ]
            )


            margin = float(
                metrics[
                    "margin"
                ]
            )


            entropy = float(
                metrics[
                    "entropy"
                ]
            )


            classifier_ms = float(
                metrics[
                    "inference_ms"
                ]
            )


            classifier_times.append(
                classifier_ms
            )


            # ======================================================================================
            # L27
            # ======================================================================================

            l27_prediction = (
                l27_reject(
                    metrics,
                    hand_ratio
                )
            )


            if (
                l27_prediction
                ==
                "NO_SIGN"
            ):

                l27_rejected += 1


            else:

                l27_accepted += 1


            # ======================================================================================
            # L28
            # ======================================================================================

            stable_prediction,changed = (
                stabilizer.update(
                    l27_prediction
                )
            )


            if changed:

                stable_changes += 1


                event = {

                    "time_seconds":
                        float(
                            time.perf_counter()
                            -
                            session_start
                        ),

                    "frame":
                        int(
                            frame_count
                        ),

                    "stable_prediction":
                        stable_prediction,

                    "raw_prediction":
                        raw_prediction,

                    "l27_prediction":
                        l27_prediction,

                    "confidence":
                        confidence,

                    "margin":
                        margin,

                    "entropy":
                        entropy,

                    "hand_ratio":
                        hand_ratio
                }


                stable_events.append(
                    event
                )


                print(
                    f"[L28 STABLE] "
                    f"{stable_prediction.upper()} "
                    f"| raw={raw_prediction} "
                    f"| L27={l27_prediction} "
                    f"| conf={confidence:.3f}"
                )


        # ==========================================================================================
        # DRAW
        # ==========================================================================================

        display = (
            frame.copy()
        )


        for result in results:

            try:

                draw_hand(
                    display,
                    result
                )

            except Exception:

                pass


        # ==========================================================================================
        # FPS
        # ==========================================================================================

        fps_frames += 1


        now = (
            time.perf_counter()
        )


        fps_elapsed = (
            now -
            fps_start
        )


        if fps_elapsed >= 0.5:

            live_fps = (

                fps_frames /
                fps_elapsed
            )


            fps_frames = 0

            fps_start = now


        # ==========================================================================================
        # OVERLAY
        # ==========================================================================================

        overlay = (
            display.copy()
        )


        cv2.rectangle(
            overlay,
            (
                0,
                0
            ),
            (
                display.shape[1],
                260
            ),
            (
                0,
                0,
                0
            ),
            -1
        )


        cv2.addWeighted(
            overlay,
            0.68,
            display,
            0.32,
            0,
            display
        )


        # ==========================================================================================
        # STABLE COLOR
        # ==========================================================================================

        if (
            stable_prediction
            ==
            "NO_SIGN"
        ):

            stable_color = (
                0,
                165,
                255
            )


        else:

            stable_color = (
                0,
                255,
                0
            )


        # ==========================================================================================
        # TEXT
        # ==========================================================================================

        cv2.putText(
            display,
            (
                f"L28 | FPS "
                f"{live_fps:.1f}"
            ),
            (
                10,
                24
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                "STABLE: "
                +
                stable_prediction.upper()
            ),
            (
                10,
                62
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.85,
            stable_color,
            2,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                "L27: "
                +
                l27_prediction.upper()
            ),
            (
                10,
                92
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                "RAW: "
                +
                raw_prediction.upper()
            ),
            (
                10,
                118
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Conf {confidence:.3f} | "
                f"Margin {margin:.3f} | "
                f"Entropy {entropy:.3f}"
            ),
            (
                10,
                144
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.39,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Hand {hand_ratio:.2f} | "
                f"Buffer "
                f"{len(feature_buffer)}/32"
            ),
            (
                10,
                170
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.39,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        history_text = " ".join(

            "N"

            if x == "NO_SIGN"

            else x[:3].upper()

            for x in stabilizer.history
        )


        cv2.putText(
            display,
            (
                "L28 history: "
                +
                history_text
            ),
            (
                10,
                196
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.37,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Palm {palm_ms:.1f}ms | "
                f"LM {landmark_ms:.1f}ms | "
                f"GRU {classifier_ms:.2f}ms"
            ),
            (
                10,
                222
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.37,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                "Q=QUIT | R=RESET"
            ),
            (
                10,
                246
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.36,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # SHOW
        # ==========================================================================================

        cv2.imshow(
            "CISLR L28 - Temporal Stabilization",
            display
        )


        # ==========================================================================================
        # KEY
        # ==========================================================================================

        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        if (
            key == ord("q")
            or
            key == 27
        ):

            break


        elif key == ord("r"):

            feature_buffer.clear()

            hand_buffer.clear()


            stabilizer.reset()


            last_feature = None


            raw_prediction = "-"

            l27_prediction = (
                "NO_SIGN"
            )

            stable_prediction = (
                "NO_SIGN"
            )


            confidence = 0.0

            margin = 0.0

            entropy = 1.0

            classifier_ms = 0.0


            print()
            print(
                "[RESET] L28 temporal state cleared."
            )


        # ==========================================================================================
        # FRAME TIME
        # ==========================================================================================

        frame_times.append(

            (
                time.perf_counter()
                -
                frame_start
            )
            *
            1000.0
        )


finally:

    cap.release()

    cv2.destroyAllWindows()


# ==================================================================================================
# 38. SESSION SUMMARY
# ==================================================================================================

duration = (
    time.perf_counter()
    -
    session_start
)


overall_fps = (

    frame_count /
    duration

    if duration > 0

    else 0.0
)


hand_detection_rate = (

    hand_frames /
    frame_count

    if frame_count > 0

    else 0.0
)


def safe_mean(
    values
):

    if not values:

        return None


    return float(
        np.mean(
            values
        )
    )


def safe_median(
    values
):

    if not values:

        return None


    return float(
        np.median(
            values
        )
    )


# ==================================================================================================
# 39. SAVE EVENTS
# ==================================================================================================

EVENT_PATH = (
    REPORT_ROOT /
    "l28_stable_events.json"
)


with open(
    EVENT_PATH,
    "w"
) as f:

    json.dump(
        stable_events,
        f,
        indent=2
    )


# ==================================================================================================
# 40. SAVE CONFIG
# ==================================================================================================

CONFIG_PATH = (
    CONFIG_ROOT /
    "l28_config.json"
)


l28_config = {

    "stage":
        "L28",

    "sequence_length":
        SEQUENCE_LENGTH,

    "feature_dim":
        FEATURE_DIM,

    "stability_window":
        STABILITY_WINDOW,

    "required_votes":
        REQUIRED_VOTES,

    "required_ratio":
        REQUIRED_RATIO,

    "change_votes":
        CHANGE_VOTES,

    "release_nosign":
        RELEASE_NO_SIGN,

    "l27_thresholds": {

        "confidence":
            CONF_T,

        "margin":
            MARGIN_T,

        "entropy":
            ENTROPY_T,

        "hand_ratio":
            HAND_T
    },

    "locked_test_used":
        False,

    "videomae_used":
        False,

    "sentence_generation":
        False
}


with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        l28_config,
        f,
        indent=2
    )


# ==================================================================================================
# 41. SAVE SUMMARY
# ==================================================================================================

SUMMARY_PATH = (
    REPORT_ROOT /
    "l28_session_summary.json"
)


summary = {

    "stage":
        "L28",

    "status":
        "SESSION_COMPLETE",

    "frames":
        int(
            frame_count
        ),

    "duration_seconds":
        float(
            duration
        ),

    "overall_fps":
        float(
            overall_fps
        ),

    "hand_frames":
        int(
            hand_frames
        ),

    "hand_detection_rate":
        float(
            hand_detection_rate
        ),

    "classification_windows":
        int(
            classification_windows
        ),

    "l27_accepted_windows":
        int(
            l27_accepted
        ),

    "l27_rejected_windows":
        int(
            l27_rejected
        ),

    "stable_state_changes":
        int(
            stable_changes
        ),

    "mean_frame_ms":
        safe_mean(
            frame_times
        ),

    "median_frame_ms":
        safe_median(
            frame_times
        ),

    "mean_palm_ms":
        safe_mean(
            palm_times
        ),

    "median_palm_ms":
        safe_median(
            palm_times
        ),

    "mean_landmark_ms":
        safe_mean(
            landmark_times
        ),

    "median_landmark_ms":
        safe_median(
            landmark_times
        ),

    "mean_classifier_ms":
        safe_mean(
            classifier_times
        ),

    "median_classifier_ms":
        safe_median(
            classifier_times
        ),

    "locked_test_used":
        False,

    "videomae_used":
        False,

    "l29_sentence_generation":
        False
}


with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ==================================================================================================
# 42. MASTER RECORD
# ==================================================================================================

MASTER_PATH = (
    L28_ROOT /
    "L28_TEMPORAL_STABILIZATION_MASTER.json"
)


master = {

    "stage":
        "L28",

    "status":
        "DEVELOPMENT_RUN_COMPLETE",

    "description":
        (
            "Real-time temporal stabilization "
            "on top of L27 rejection."
        ),

    "input_pipeline":
        (
            "OpenVINO palm -> "
            "OpenVINO landmarks -> "
            "L26C BiGRU -> "
            "L27 rejection"
        ),

    "output":
        (
            "temporally stabilized "
            "10-class sign or NO_SIGN"
        ),

    "stabilization": {

        "window":
            STABILITY_WINDOW,

        "initial_votes":
            REQUIRED_VOTES,

        "ratio":
            REQUIRED_RATIO,

        "change_votes":
            CHANGE_VOTES,

        "release_nosign":
            RELEASE_NO_SIGN
    },

    "artifacts": {

        "config":
            str(
                CONFIG_PATH
            ),

        "summary":
            str(
                SUMMARY_PATH
            ),

        "events":
            str(
                EVENT_PATH
            )
    },

    "integrity": {

        "locked_test_used":
            False,

        "training":
            False,

        "retraining":
            False,

        "videomae":
            False,

        "l29_sentence_generation":
            False
    }
}


with open(
    MASTER_PATH,
    "w"
) as f:

    json.dump(
        master,
        f,
        indent=2
    )


# ==================================================================================================
# 43. HASH MASTER
# ==================================================================================================

with open(
    MASTER_PATH,
    "rb"
) as f:

    master_sha = (
        hashlib.sha256(
            f.read()
        ).hexdigest()
    )


# ==================================================================================================
# 44. FINAL OUTPUT
# ==================================================================================================

print()
print("=" * 120)
print("L28 — TEMPORAL STABILIZATION SESSION COMPLETE")
print("=" * 120)

print()

print(
    "Frames                  :",
    frame_count
)

print(
    "Duration                :",
    f"{duration:.2f} sec"
)

print(
    "Overall FPS             :",
    f"{overall_fps:.2f}"
)

print()

print(
    "Hand frames             :",
    hand_frames
)

print(
    "Hand detection rate     :",
    f"{hand_detection_rate*100:.2f}%"
)

print()

print(
    "Classification windows  :",
    classification_windows
)

print(
    "L27 accepted windows    :",
    l27_accepted
)

print(
    "L27 rejected windows    :",
    l27_rejected
)

print(
    "Stable state changes    :",
    stable_changes
)

print()

if frame_times:

    print(
        "Mean frame time         :",
        f"{np.mean(frame_times):.3f} ms"
    )

    print(
        "Median frame time       :",
        f"{np.median(frame_times):.3f} ms"
    )


if palm_times:

    print(
        "Mean palm time          :",
        f"{np.mean(palm_times):.3f} ms"
    )


if landmark_times:

    print(
        "Mean landmark time      :",
        f"{np.mean(landmark_times):.3f} ms"
    )


if classifier_times:

    print(
        "Mean BiGRU time         :",
        f"{np.mean(classifier_times):.3f} ms"
    )

    print(
        "Median BiGRU time       :",
        f"{np.median(classifier_times):.3f} ms"
    )


print()
print("-" * 120)

print(
    "Config:"
)

print(
    CONFIG_PATH
)

print()

print(
    "Summary:"
)

print(
    SUMMARY_PATH
)

print()

print(
    "Stable events:"
)

print(
    EVENT_PATH
)

print()

print(
    "Master:"
)

print(
    MASTER_PATH
)

print()

print(
    "Master SHA256:"
)

print(
    master_sha
)

print("-" * 120)

print()

print(
    "Locked test used        : NO"
)

print(
    "Training/retraining     : NO"
)

print(
    "VideoMAE used           : NO"
)

print(
    "L27 rejection           : YES"
)

print(
    "L28 stabilization       : YES"
)

print(
    "L29 sentence generation : NO"
)

print()

print(
    "[PASS] L28 development run complete."
)

CISLR — L28 FULL STANDALONE INITIALIZATION
Palm detector       : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/palm/palm_detection_full.tflite
Hand landmarks      : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/hand_landmark/hand_landmark_full.tflite
L26C BiGRU          : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l26c_realtime_landmark_classifier/models/l26c_fast_landmark_bigru.xml
L27 thresholds      : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l27_nosign_rejection/config/l27_rejection_thresholds.json

[PASS] All required files found.

L27 THRESHOLDS

Confidence >= 0.100
Margin     >= 0.000
Entropy    <= 0.800
Hand ratio >= 0.800

OpenVINO: 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Devices : ['CPU', 'GPU']

Loading palm detector...
[PASS] Palm: [1,192,192,3]
Loading landmark detector...
[PASS] Landmark: [1,224,224,3]
Loading temporal classifier...
[PASS] BiGRU input 

QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or sw

[L28 STABLE] PENCIL | raw=pencil | L27=pencil | conf=0.324
[L28 STABLE] TUESDAY | raw=tuesday | L27=tuesday | conf=0.358
[L28 STABLE] FRIDAY | raw=friday | L27=friday | conf=0.488
[L28 STABLE] TUESDAY | raw=tuesday | L27=tuesday | conf=0.594
[L28 STABLE] MOTHER | raw=mother | L27=mother | conf=0.652
[L28 STABLE] THURSDAY | raw=thursday | L27=thursday | conf=0.481
[L28 STABLE] MONDAY | raw=monday | L27=monday | conf=0.393
[L28 STABLE] MOTHER | raw=mother | L27=mother | conf=0.797
[L28 STABLE] MONDAY | raw=monday | L27=monday | conf=0.422
[L28 STABLE] THURSDAY | raw=thursday | L27=thursday | conf=0.444
[L28 STABLE] MONDAY | raw=monday | L27=monday | conf=0.401
[L28 STABLE] PENCIL | raw=pencil | L27=pencil | conf=0.572
[L28 STABLE] TUESDAY | raw=tuesday | L27=tuesday | conf=0.400
[L28 STABLE] THURSDAY | raw=thursday | L27=thursday | conf=0.703
[L28 STABLE] MOTHER | raw=mother | L27=mother | conf=0.434
[L28 STABLE] WASH | raw=wash | L27=wash | conf=0.574
[L28 STABLE] NO_SIGN | raw=wash | L

In [4]:
# ==================================================================================================
# CISLR — L29 CORRECTED
# MODEL-DIRECT SIGN OUTPUT + L27 ONLY FOR NO_SIGN
# ==================================================================================================
#
# RULE:
#
#   L26C trained model predicts the sign.
#
#       model = mother
#
#   L27 is ONLY used to decide:
#
#       valid sign OR NO_SIGN
#
#   Therefore:
#
#       model=MOTHER + L27 valid   -> MOTHER
#       model=MOTHER + L27 reject  -> NO_SIGN
#
#   There is NO "stable class" capable of changing MOTHER into TUESDAY.
#
#
# Sentence logic:
#
#   repeated MODEL prediction
#           ↓
#   confirm event
#           ↓
#   append model label ONCE
#
#   NO_SIGN releases the event so the next sign can be appended.
#
#
# Controls:
#
#   Q / ESC = Quit
#   C       = Clear sentence
#   R       = Reset recognition buffers
#   B       = Delete last token
#   S       = Print sentence
#
# ==================================================================================================

from pathlib import Path
from collections import deque, Counter
import cv2
import json
import time
import hashlib
import numpy as np


# ==================================================================================================
# 1. CHECK REQUIRED FUNCTIONS FROM FULL L28 CELL
# ==================================================================================================

required = [
    "detect_hands",
    "draw_hand",
    "normalize_for_classifier",
    "classify_sequence",
    "l27_reject",
    "CLASS_NAMES",
    "SEQUENCE_LENGTH",
    "FEATURE_DIM"
]


missing = [
    x
    for x in required
    if x not in globals()
]


if missing:

    raise RuntimeError(
        "\nRun the FULL L28 standalone cell first and quit its webcam with Q.\n\n"
        "Missing:\n"
        +
        "\n".join(missing)
    )


# ==================================================================================================
# 2. OUTPUT PATHS
# ==================================================================================================

LANDMARK_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)


L29_ROOT = (
    LANDMARK_ROOT /
    "audit/l29_model_direct_sentence"
)


REPORT_ROOT = (
    L29_ROOT /
    "reports"
)


CONFIG_ROOT = (
    L29_ROOT /
    "config"
)


REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 3. CAMERA
# ==================================================================================================

CAMERA_ID = 0

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480


# ==================================================================================================
# 4. MODEL-CONFIRMATION SETTINGS
# ==================================================================================================
#
# IMPORTANT:
#
# These settings DO NOT select another class.
#
# They only ask:
#
#   "Has the MODEL itself predicted the same label enough times?"
#
# Example:
#
#   MODEL:
#       mother mother mother mother
#
#   confirmed:
#       mother
#
#
# They can NEVER turn:
#
#       mother -> tuesday
#
# unless the trained model itself starts predicting tuesday.
#
# ==================================================================================================

MODEL_HISTORY_SIZE = 5

MODEL_CONFIRM_VOTES = 3


# How many consecutive NO_SIGN outputs release current token.

NO_SIGN_RELEASE_FRAMES = 3


MAX_SENTENCE_TOKENS = 30


# ==================================================================================================
# 5. TEMPORAL BUFFERS
# ==================================================================================================

feature_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


hand_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


model_history = deque(
    maxlen=MODEL_HISTORY_SIZE
)


last_feature = None


# ==================================================================================================
# 6. SENTENCE STATE
# ==================================================================================================

sentence_tokens = []


token_events = []


active_token = None


nosign_count = 0


# ==================================================================================================
# 7. DISPLAY STATE
# ==================================================================================================

model_prediction = "-"

output_prediction = "NO_SIGN"


confidence = 0.0

margin = 0.0

entropy = 1.0


classifier_ms = 0.0


hand_ratio = 0.0


# ==================================================================================================
# 8. SESSION METRICS
# ==================================================================================================

frame_count = 0

hand_frames = 0

classification_windows = 0


valid_windows = 0

nosign_windows = 0


tokens_added = 0


session_start = time.perf_counter()


fps_start = time.perf_counter()

fps_frames = 0

live_fps = 0.0


# ==================================================================================================
# 9. MODEL CONFIRMATION
# ==================================================================================================

def get_confirmed_model_prediction():

    """
    IMPORTANT:

    Uses ONLY predictions generated by the trained L26C model.

    It cannot invent/change a class.

    Returns:
        class name
        or None
    """

    if len(model_history) < MODEL_CONFIRM_VOTES:

        return None


    counts = Counter(
        model_history
    )


    candidate, votes = (
        counts.most_common(1)[0]
    )


    if votes >= MODEL_CONFIRM_VOTES:

        return candidate


    return None


# ==================================================================================================
# 10. CAMERA
# ==================================================================================================

cap = cv2.VideoCapture(
    CAMERA_ID
)


if not cap.isOpened():

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    CAMERA_WIDTH
)


cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    CAMERA_HEIGHT
)


try:

    cap.set(
        cv2.CAP_PROP_BUFFERSIZE,
        1
    )

except Exception:

    pass


print("=" * 110)
print("L29 — MODEL-DIRECT SIGN SEQUENCE")
print("=" * 110)

print()

print(
    "SIGN SOURCE : L26C TRAINED MODEL"
)

print(
    "L27 ROLE    : NO_SIGN ONLY"
)

print(
    "L28 CLASS   : NOT USED TO CHANGE MODEL CLASS"
)

print()

print(
    "Vocabulary:"
)

print(
    CLASS_NAMES
)

print()

print(
    "Q/ESC = quit"
)

print(
    "C     = clear sentence"
)

print(
    "R     = reset recognition"
)

print(
    "B     = delete last token"
)

print(
    "S     = print sentence"
)

print()


# ==================================================================================================
# 11. LIVE LOOP
# ==================================================================================================

try:

    while True:

        ok, frame = cap.read()


        if not ok:

            continue


        frame_count += 1


        # ==========================================================================================
        # HAND DETECTION
        # ==========================================================================================

        try:

            results, palm_ms, landmark_ms = detect_hands(
                frame
            )

        except Exception:

            results = []

            palm_ms = 0.0

            landmark_ms = 0.0


        current_feature = None

        hand_present = False


        # ==========================================================================================
        # EXTRACT LANDMARK FEATURE
        # ==========================================================================================

        if results:

            current_feature = normalize_for_classifier(
                results[0]["landmarks"]
            )


            if current_feature is not None:

                hand_present = True

                hand_frames += 1


                last_feature = (
                    current_feature.copy()
                )


        # ==========================================================================================
        # HAND EVIDENCE
        # ==========================================================================================

        hand_buffer.append(
            int(hand_present)
        )


        hand_ratio = float(
            np.mean(
                hand_buffer
            )
        ) if hand_buffer else 0.0


        # ==========================================================================================
        # TEMPORAL MODEL INPUT
        # ==========================================================================================

        if current_feature is not None:

            feature_buffer.append(
                current_feature.copy()
            )


        elif last_feature is not None:

            feature_buffer.append(
                last_feature.copy()
            )


        # ==========================================================================================
        # DEFAULT OUTPUT
        # ==========================================================================================

        output_prediction = "NO_SIGN"


        # ==========================================================================================
        # TRAINED MODEL
        # ==========================================================================================

        if len(feature_buffer) == SEQUENCE_LENGTH:

            metrics = classify_sequence(
                np.stack(
                    feature_buffer
                )
            )


            classification_windows += 1


            # --------------------------------------------------------------------------------------
            # THIS IS THE AUTHORITATIVE SIGN LABEL
            # --------------------------------------------------------------------------------------

            model_prediction = (
                metrics["prediction"]
            )


            confidence = float(
                metrics["confidence"]
            )


            margin = float(
                metrics["margin"]
            )


            entropy = float(
                metrics["entropy"]
            )


            classifier_ms = float(
                metrics["inference_ms"]
            )


            # ======================================================================================
            # L27
            #
            # IMPORTANT:
            #
            # We DO NOT take the sign label from L27.
            #
            # We only ask whether L27 rejected the window.
            # ======================================================================================

            rejection_result = l27_reject(
                metrics,
                hand_ratio
            )


            rejected_as_nosign = (
                rejection_result
                ==
                "NO_SIGN"
            )


            # ======================================================================================
            # NO_SIGN
            # ======================================================================================

            if rejected_as_nosign:

                output_prediction = (
                    "NO_SIGN"
                )


                nosign_windows += 1


                nosign_count += 1


                # Do not let old model predictions leak into next sign.

                model_history.clear()


                # ----------------------------------------------------------------------------------
                # RELEASE CURRENT TOKEN
                # ----------------------------------------------------------------------------------

                if (
                    nosign_count
                    >=
                    NO_SIGN_RELEASE_FRAMES
                ):

                    active_token = None


            # ======================================================================================
            # VALID SIGN
            # ======================================================================================

            else:

                valid_windows += 1


                nosign_count = 0


                # ----------------------------------------------------------------------------------
                # OUTPUT = MODEL
                #
                # NO STABLE CLASS HERE.
                # ----------------------------------------------------------------------------------

                output_prediction = (
                    model_prediction
                )


                # ----------------------------------------------------------------------------------
                # History contains MODEL predictions only.
                # ----------------------------------------------------------------------------------

                model_history.append(
                    model_prediction
                )


                confirmed_model_sign = (
                    get_confirmed_model_prediction()
                )


                # ==================================================================================
                # NEW TOKEN EVENT
                # ==================================================================================

                if confirmed_model_sign is not None:

                    # ------------------------------------------------------------------------------
                    # No token currently active.
                    # ------------------------------------------------------------------------------

                    if active_token is None:

                        active_token = (
                            confirmed_model_sign
                        )


                        sentence_tokens.append(
                            confirmed_model_sign
                        )


                        if (
                            len(sentence_tokens)
                            >
                            MAX_SENTENCE_TOKENS
                        ):

                            sentence_tokens = (
                                sentence_tokens[
                                    -MAX_SENTENCE_TOKENS:
                                ]
                            )


                        tokens_added += 1


                        event = {

                            "event_index":
                                int(tokens_added),

                            "time_seconds":
                                float(
                                    time.perf_counter()
                                    -
                                    session_start
                                ),

                            "frame":
                                int(frame_count),

                            "token":
                                confirmed_model_sign,

                            "model_prediction":
                                model_prediction,

                            "confidence":
                                confidence,

                            "margin":
                                margin,

                            "entropy":
                                entropy,

                            "hand_ratio":
                                hand_ratio,

                            "sentence":
                                " ".join(
                                    sentence_tokens
                                )
                        }


                        token_events.append(
                            event
                        )


                        print()

                        print(
                            "[MODEL TOKEN]",
                            confirmed_model_sign.upper()
                        )


                        print(
                            "[SEQUENCE]",
                            " ".join(
                                sentence_tokens
                            )
                        )


                    # ------------------------------------------------------------------------------
                    # SAME active token:
                    # do nothing.
                    #
                    # Holding MOTHER should NOT generate:
                    #
                    # mother mother mother mother
                    # ------------------------------------------------------------------------------

                    elif (
                        confirmed_model_sign
                        ==
                        active_token
                    ):

                        pass


                    # ------------------------------------------------------------------------------
                    # DIFFERENT model prediction while previous token still active.
                    #
                    # Do NOT replace/append it until NO_SIGN releases the previous event.
                    #
                    # This protects against transition noise.
                    # ------------------------------------------------------------------------------

                    else:

                        pass


        # ==========================================================================================
        # DRAW HAND
        # ==========================================================================================

        display = frame.copy()


        for result in results:

            try:

                draw_hand(
                    display,
                    result
                )

            except Exception:

                pass


        # ==========================================================================================
        # FPS
        # ==========================================================================================

        fps_frames += 1


        now = time.perf_counter()


        fps_elapsed = (
            now -
            fps_start
        )


        if fps_elapsed >= 0.5:

            live_fps = (
                fps_frames /
                fps_elapsed
            )


            fps_frames = 0

            fps_start = now


        # ==========================================================================================
        # DISPLAY BACKGROUND
        # ==========================================================================================

        overlay = display.copy()


        cv2.rectangle(
            overlay,
            (0,0),
            (
                display.shape[1],
                300
            ),
            (0,0,0),
            -1
        )


        cv2.addWeighted(
            overlay,
            0.68,
            display,
            0.32,
            0,
            display
        )


        # ==========================================================================================
        # OUTPUT COLOR
        # ==========================================================================================

        if output_prediction == "NO_SIGN":

            output_color = (
                0,
                165,
                255
            )

        else:

            output_color = (
                0,
                255,
                0
            )


        # ==========================================================================================
        # UI
        # ==========================================================================================

        cv2.putText(
            display,
            f"L29 | FPS {live_fps:.1f}",
            (10,24),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        # ------------------------------------------------------------------------------------------
        # TRAINED MODEL
        # ------------------------------------------------------------------------------------------

        cv2.putText(
            display,
            (
                "MODEL: "
                +
                model_prediction.upper()
            ),
            (10,60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.82,
            (255,255,255),
            2,
            cv2.LINE_AA
        )


        # ------------------------------------------------------------------------------------------
        # FINAL OUTPUT
        # ------------------------------------------------------------------------------------------

        cv2.putText(
            display,
            (
                "OUTPUT: "
                +
                output_prediction.upper()
            ),
            (10,98),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.82,
            output_color,
            2,
            cv2.LINE_AA
        )


        # ------------------------------------------------------------------------------------------
        # ACTIVE TOKEN
        # ------------------------------------------------------------------------------------------

        active_text = (
            active_token.upper()
            if active_token is not None
            else "NONE"
        )


        cv2.putText(
            display,
            (
                "ACTIVE TOKEN: "
                +
                active_text
            ),
            (10,127),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.44,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Conf {confidence:.3f} | "
                f"Margin {margin:.3f} | "
                f"Entropy {entropy:.3f}"
            ),
            (10,153),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.38,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Hand {hand_ratio:.2f} | "
                f"NO_SIGN count {nosign_count}/{NO_SIGN_RELEASE_FRAMES}"
            ),
            (10,179),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.38,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Model votes: "
                f"{list(model_history)}"
            )[-85:],
            (10,205),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.34,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # SENTENCE
        # ==========================================================================================

        sentence = " ".join(
            sentence_tokens
        )


        if not sentence:

            sentence = (
                "<waiting for sign>"
            )


        display_sentence = (
            sentence[-75:]
        )


        cv2.putText(
            display,
            "SEQUENCE:",
            (10,235),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.43,
            (255,255,0),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            display_sentence,
            (10,262),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.52,
            (0,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            "Q quit | C clear | R reset | B delete | S print",
            (10,288),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.34,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.imshow(
            "CISLR L29 - Model Direct Sequence",
            display
        )


        # ==========================================================================================
        # KEYBOARD
        # ==========================================================================================

        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        # ------------------------------------------------------------------------------------------
        # QUIT
        # ------------------------------------------------------------------------------------------

        if (
            key == ord("q")
            or
            key == 27
        ):

            break


        # ------------------------------------------------------------------------------------------
        # CLEAR SENTENCE
        # ------------------------------------------------------------------------------------------

        elif key == ord("c"):

            sentence_tokens.clear()

            token_events.clear()


            active_token = None

            nosign_count = 0


            model_history.clear()

            feature_buffer.clear()

            hand_buffer.clear()


            last_feature = None


            print(
                "[CLEAR] Sentence cleared."
            )


        # ------------------------------------------------------------------------------------------
        # RESET RECOGNITION
        # ------------------------------------------------------------------------------------------

        elif key == ord("r"):

            active_token = None

            nosign_count = 0


            model_history.clear()

            feature_buffer.clear()

            hand_buffer.clear()


            last_feature = None


            model_prediction = "-"

            output_prediction = (
                "NO_SIGN"
            )


            confidence = 0.0

            margin = 0.0

            entropy = 1.0


            print(
                "[RESET] Recognition buffers reset."
            )


        # ------------------------------------------------------------------------------------------
        # DELETE LAST TOKEN
        # ------------------------------------------------------------------------------------------

        elif key == ord("b"):

            if sentence_tokens:

                removed = (
                    sentence_tokens.pop()
                )


                print(
                    "[DELETE]",
                    removed
                )


                print(
                    "[SEQUENCE]",
                    " ".join(
                        sentence_tokens
                    )
                )


        # ------------------------------------------------------------------------------------------
        # PRINT
        # ------------------------------------------------------------------------------------------

        elif key == ord("s"):

            print()

            print(
                "[CURRENT SEQUENCE]"
            )

            print(
                " ".join(
                    sentence_tokens
                )
            )

            print()


finally:

    cap.release()

    cv2.destroyAllWindows()


# ==================================================================================================
# 12. SESSION SUMMARY
# ==================================================================================================

duration = (
    time.perf_counter()
    -
    session_start
)


overall_fps = (
    frame_count /
    duration
    if duration > 0
    else 0.0
)


hand_rate = (
    hand_frames /
    frame_count
    if frame_count > 0
    else 0.0
)


final_sequence = " ".join(
    sentence_tokens
)


# ==================================================================================================
# 13. SAVE EVENTS
# ==================================================================================================

EVENT_PATH = (
    REPORT_ROOT /
    "l29_model_token_events.json"
)


with open(
    EVENT_PATH,
    "w"
) as f:

    json.dump(
        token_events,
        f,
        indent=2
    )


# ==================================================================================================
# 14. SAVE FINAL SEQUENCE
# ==================================================================================================

SEQUENCE_PATH = (
    REPORT_ROOT /
    "l29_final_sequence.txt"
)


with open(
    SEQUENCE_PATH,
    "w"
) as f:

    f.write(
        final_sequence
    )


# ==================================================================================================
# 15. CONFIG
# ==================================================================================================

CONFIG_PATH = (
    CONFIG_ROOT /
    "l29_model_direct_config.json"
)


config = {

    "stage":
        "L29",

    "design":
        "model_direct_sign_l27_nosign_only",

    "sign_identity_source":
        "L26C_trained_BiGRU",

    "l27_role":
        "NO_SIGN_rejection_only",

    "l28_class_override":
        False,

    "model_history_size":
        MODEL_HISTORY_SIZE,

    "model_confirm_votes":
        MODEL_CONFIRM_VOTES,

    "nosign_release_frames":
        NO_SIGN_RELEASE_FRAMES,

    "sequence_length":
        SEQUENCE_LENGTH,

    "feature_dim":
        FEATURE_DIM,

    "vocabulary":
        CLASS_NAMES,

    "locked_test_used":
        False,

    "training":
        False,

    "retraining":
        False
}


with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ==================================================================================================
# 16. SUMMARY
# ==================================================================================================

SUMMARY_PATH = (
    REPORT_ROOT /
    "l29_model_direct_summary.json"
)


summary = {

    "stage":
        "L29",

    "status":
        "DEVELOPMENT_RUN_COMPLETE",

    "frames":
        int(frame_count),

    "duration_seconds":
        float(duration),

    "overall_fps":
        float(overall_fps),

    "hand_detection_rate":
        float(hand_rate),

    "classification_windows":
        int(classification_windows),

    "valid_sign_windows":
        int(valid_windows),

    "nosign_windows":
        int(nosign_windows),

    "tokens_added":
        int(tokens_added),

    "final_tokens":
        sentence_tokens,

    "final_sequence":
        final_sequence,

    "sign_identity_source":
        "trained_model",

    "nosign_source":
        "L27",

    "locked_test_used":
        False
}


with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ==================================================================================================
# 17. MASTER
# ==================================================================================================

MASTER_PATH = (
    L29_ROOT /
    "L29_MODEL_DIRECT_MASTER.json"
)


master = {

    "stage":
        "L29",

    "status":
        "DEVELOPMENT_RUN_COMPLETE",

    "pipeline": (
        "OpenVINO palm -> "
        "OpenVINO landmarks -> "
        "L26C trained BiGRU sign identity -> "
        "L27 NO_SIGN gate -> "
        "model-consistency token confirmation -> "
        "sequence"
    ),

    "critical_rule": (
        "L27 may convert a model prediction to NO_SIGN, "
        "but may never substitute another sign class."
    ),

    "artifacts": {

        "config":
            str(CONFIG_PATH),

        "summary":
            str(SUMMARY_PATH),

        "events":
            str(EVENT_PATH),

        "sequence":
            str(SEQUENCE_PATH)
    },

    "integrity": {

        "locked_test_used":
            False,

        "training":
            False,

        "retraining":
            False,

        "videomae":
            False,

        "tts":
            False
    }
}


with open(
    MASTER_PATH,
    "w"
) as f:

    json.dump(
        master,
        f,
        indent=2
    )


# ==================================================================================================
# 18. HASH
# ==================================================================================================

with open(
    MASTER_PATH,
    "rb"
) as f:

    master_sha = hashlib.sha256(
        f.read()
    ).hexdigest()


# ==================================================================================================
# 19. FINAL OUTPUT
# ==================================================================================================

print()

print("=" * 110)
print("L29 — MODEL-DIRECT SEQUENCE SESSION COMPLETE")
print("=" * 110)

print()

print(
    "Frames                 :",
    frame_count
)

print(
    "Duration               :",
    f"{duration:.2f} sec"
)

print(
    "Overall FPS            :",
    f"{overall_fps:.2f}"
)

print(
    "Hand detection rate    :",
    f"{hand_rate*100:.2f}%"
)

print()

print(
    "Classification windows :",
    classification_windows
)

print(
    "Valid sign windows     :",
    valid_windows
)

print(
    "NO_SIGN windows        :",
    nosign_windows
)

print()

print(
    "Tokens generated       :",
    tokens_added
)

print()

print(
    "FINAL TOKENS:"
)

print(
    sentence_tokens
)

print()

print(
    "FINAL SEQUENCE:"
)

print(
    final_sequence
)

print()

print("-" * 110)

print(
    "Sign source            : L26C TRAINED MODEL"
)

print(
    "NO_SIGN source         : L27"
)

print(
    "L28 class override     : DISABLED"
)

print(
    "Locked test used       : NO"
)

print(
    "Training/retraining    : NO"
)

print()

print(
    "Master:"
)

print(
    MASTER_PATH
)

print()

print(
    "Master SHA256:"
)

print(
    master_sha
)

print("-" * 110)

print()

print(
    "[PASS] L29 model-direct development run complete."
)

L29 — MODEL-DIRECT SIGN SEQUENCE

SIGN SOURCE : L26C TRAINED MODEL
L27 ROLE    : NO_SIGN ONLY
L28 CLASS   : NOT USED TO CHANGE MODEL CLASS

Vocabulary:
['monday', 'interest', 'thursday', 'christmas', 'pencil', 'mother', 'tuesday', 'friday', 'rubber', 'wash']

Q/ESC = quit
C     = clear sentence
R     = reset recognition
B     = delete last token
S     = print sentence


[MODEL TOKEN] WASH
[SEQUENCE] wash

[MODEL TOKEN] PENCIL
[SEQUENCE] wash pencil

L29 — MODEL-DIRECT SEQUENCE SESSION COMPLETE

Frames                 : 2277
Duration               : 115.63 sec
Overall FPS            : 19.69
Hand detection rate    : 94.73%

Classification windows : 2246
Valid sign windows     : 2118
NO_SIGN windows        : 128

Tokens generated       : 2

FINAL TOKENS:
['wash', 'pencil']

FINAL SEQUENCE:
wash pencil

--------------------------------------------------------------------------------------------------------------
Sign source            : L26C TRAINED MODEL
NO_SIGN source         : L27
L28 c

In [6]:
%pip install pyttsx3


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==================================================================================================
# CISLR — L30 FULL STANDALONE
# ONE CELL — SAFE AFTER KERNEL RESTART
# ==================================================================================================
#
# PIPELINE
#
# Webcam
#   ↓
# OpenVINO Palm Detector
#   ↓
# OpenVINO 21-point Hand Landmark Detector
#   ↓
# 32 × 63 landmark sequence
#   ↓
# L26C OpenVINO BiGRU
#   ↓
# MODEL decides sign identity
#   ↓
# L27 is used ONLY as NO_SIGN rejection
#   ↓
# repeated MODEL predictions confirm token
#   ↓
# L29 token sequence
#   ↓
# L30 asynchronous TTS
#
# IMPORTANT:
#
# MODEL=MOTHER + accepted -> MOTHER
# MODEL=MOTHER + rejected -> NO_SIGN
#
# L27 NEVER changes MOTHER to TUESDAY.
#
# Controls:
# Q / ESC = quit
# C       = clear sentence
# R       = reset recognition
# B       = delete last token
# S       = speak complete sequence
# M       = mute/unmute
#
# ==================================================================================================

from pathlib import Path
from collections import deque, Counter
import cv2
import numpy as np
import openvino as ov
import json
import math
import time
import threading
import queue
import hashlib


# ==================================================================================================
# 0. TTS
# ==================================================================================================

try:
    import pyttsx3
except ImportError:
    raise RuntimeError(
        "\npyttsx3 is not installed.\n"
        "Open a terminal and run:\n\n"
        "conda activate openvino_face\n"
        "python -m pip install pyttsx3\n"
        "sudo apt install -y espeak-ng\n"
    )


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)

PALM_MODEL_PATH = (
    ROOT /
    "models/pretrained/palm/palm_detection_full.tflite"
)

LANDMARK_MODEL_PATH = (
    ROOT /
    "models/pretrained/hand_landmark/hand_landmark_full.tflite"
)

CLASSIFIER_XML = (
    ROOT /
    "audit/l26c_realtime_landmark_classifier/models/"
    "l26c_fast_landmark_bigru.xml"
)

L27_THRESHOLD_PATH = (
    ROOT /
    "audit/l27_nosign_rejection/config/"
    "l27_rejection_thresholds.json"
)

L30_ROOT = (
    ROOT /
    "audit/l30_text_to_speech"
)

REPORT_ROOT = L30_ROOT / "reports"
CONFIG_ROOT = L30_ROOT / "config"

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)


# ==================================================================================================
# 2. VERIFY FILES
# ==================================================================================================

required_files = [
    PALM_MODEL_PATH,
    LANDMARK_MODEL_PATH,
    CLASSIFIER_XML,
    L27_THRESHOLD_PATH
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file:\n{p}")

print("[PASS] Required model/config files found.")


# ==================================================================================================
# 3. BASIC CONFIG
# ==================================================================================================

DEVICE = "CPU"

PALM_SIZE = 192
LANDMARK_SIZE = 224

PALM_THRESHOLD = 0.50
NMS_THRESHOLD = 0.30

MAX_HANDS = 2

SEQUENCE_LENGTH = 32
FEATURE_DIM = 63

CLASS_NAMES = [
    "monday",
    "interest",
    "thursday",
    "christmas",
    "pencil",
    "mother",
    "tuesday",
    "friday",
    "rubber",
    "wash"
]

CAMERA_ID = 0
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480


# ==================================================================================================
# 4. L29/L30 SETTINGS
# ==================================================================================================

MODEL_HISTORY_SIZE = 5
MODEL_CONFIRM_VOTES = 3

NO_SIGN_RELEASE_FRAMES = 3

MAX_SENTENCE_TOKENS = 30

TTS_RATE = 150
TTS_VOLUME = 1.0

AUTO_SPEAK_NEW_TOKEN = True


# ==================================================================================================
# 5. LOAD L27 THRESHOLDS
# ==================================================================================================

with open(L27_THRESHOLD_PATH, "r") as f:
    l27_cfg = json.load(f)

print("[INFO] L27 config loaded:")
print(L27_THRESHOLD_PATH)


def find_threshold(d, possible_names, default):
    for name in possible_names:
        if name in d:
            return float(d[name])

    for v in d.values():
        if isinstance(v, dict):
            found = find_threshold(v, possible_names, None)
            if found is not None:
                return found

    return default


CONF_THRESHOLD = find_threshold(
    l27_cfg,
    [
        "confidence_threshold",
        "confidence",
        "conf_threshold",
        "min_confidence"
    ],
    0.10
)

MARGIN_THRESHOLD = find_threshold(
    l27_cfg,
    [
        "margin_threshold",
        "margin",
        "min_margin"
    ],
    0.00
)

ENTROPY_THRESHOLD = find_threshold(
    l27_cfg,
    [
        "entropy_threshold",
        "entropy",
        "max_entropy"
    ],
    0.80
)

HAND_RATIO_THRESHOLD = find_threshold(
    l27_cfg,
    [
        "hand_ratio_threshold",
        "hand_ratio",
        "min_hand_ratio"
    ],
    0.80
)


print()
print("L27 thresholds")
print("----------------------------")
print("Confidence >=", CONF_THRESHOLD)
print("Margin     >=", MARGIN_THRESHOLD)
print("Entropy    <=", ENTROPY_THRESHOLD)
print("Hand ratio >=", HAND_RATIO_THRESHOLD)
print()


# ==================================================================================================
# 6. OPENVINO
# ==================================================================================================

core = ov.Core()

print("[INFO] OpenVINO:", ov.__version__)
print("[INFO] Devices:", core.available_devices)


# ==================================================================================================
# 7. LOAD PALM MODEL
# ==================================================================================================

palm_model = core.read_model(str(PALM_MODEL_PATH))
palm_compiled = core.compile_model(palm_model, DEVICE)

palm_input = palm_compiled.input(0)

print("[PASS] Palm detector loaded.")
print("       input:", palm_input.shape)


# ==================================================================================================
# 8. LOAD LANDMARK MODEL
# ==================================================================================================

landmark_model = core.read_model(str(LANDMARK_MODEL_PATH))
landmark_compiled = core.compile_model(
    landmark_model,
    DEVICE
)

landmark_input = landmark_compiled.input(0)

print("[PASS] Hand landmark model loaded.")
print("       input:", landmark_input.shape)


# ==================================================================================================
# 9. LOAD L26C CLASSIFIER
# ==================================================================================================

classifier_model = core.read_model(
    str(CLASSIFIER_XML)
)

classifier_compiled = core.compile_model(
    classifier_model,
    DEVICE
)

classifier_input = classifier_compiled.input(0)
classifier_output = classifier_compiled.output(0)

print("[PASS] L26C BiGRU loaded.")
print("       input :", classifier_input.shape)
print("       output:", classifier_output.shape)


# ==================================================================================================
# 10. MODEL LAYOUT HELPERS
# ==================================================================================================

def get_layout(shape):

    shape = list(shape)

    if len(shape) != 4:
        raise RuntimeError(
            f"Expected 4D image input, got {shape}"
        )

    # NHWC
    if shape[-1] == 3:
        return "NHWC"

    # NCHW
    if shape[1] == 3:
        return "NCHW"

    raise RuntimeError(
        f"Cannot determine image layout from {shape}"
    )


PALM_LAYOUT = get_layout(palm_input.shape)
LANDMARK_LAYOUT = get_layout(landmark_input.shape)

print("[INFO] Palm layout    :", PALM_LAYOUT)
print("[INFO] Landmark layout:", LANDMARK_LAYOUT)


# ==================================================================================================
# 11. PALM ANCHORS
# ==================================================================================================

def generate_palm_anchors():

    anchors = []

    strides = [8, 16, 16, 16]

    num_layers = len(strides)

    layer_id = 0

    while layer_id < num_layers:

        last_same_stride_layer = layer_id

        while (
            last_same_stride_layer < num_layers
            and
            strides[last_same_stride_layer] == strides[layer_id]
        ):
            last_same_stride_layer += 1

        stride = strides[layer_id]

        feature_map_h = int(
            math.ceil(PALM_SIZE / stride)
        )

        feature_map_w = int(
            math.ceil(PALM_SIZE / stride)
        )

        anchors_per_cell = (
            2 *
            (
                last_same_stride_layer
                -
                layer_id
            )
        )

        for y in range(feature_map_h):

            for x in range(feature_map_w):

                x_center = (
                    x + 0.5
                ) / feature_map_w

                y_center = (
                    y + 0.5
                ) / feature_map_h

                for _ in range(anchors_per_cell):

                    anchors.append([
                        x_center,
                        y_center,
                        1.0,
                        1.0
                    ])

        layer_id = last_same_stride_layer

    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )

    if anchors.shape != (2016, 4):
        raise RuntimeError(
            f"Palm anchors incorrect: {anchors.shape}"
        )

    return anchors


PALM_ANCHORS = generate_palm_anchors()

print("[PASS] Palm anchors:", PALM_ANCHORS.shape)


# ==================================================================================================
# 12. SIGMOID
# ==================================================================================================

def sigmoid(x):

    x = np.clip(
        x,
        -80.0,
        80.0
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


# ==================================================================================================
# 13. LETTERBOX
# ==================================================================================================

def letterbox(image, size=192):

    h, w = image.shape[:2]

    scale = min(
        size / w,
        size / h
    )

    nw = int(round(w * scale))
    nh = int(round(h * scale))

    resized = cv2.resize(
        image,
        (nw, nh)
    )

    canvas = np.zeros(
        (size, size, 3),
        dtype=np.uint8
    )

    left = (
        size - nw
    ) // 2

    top = (
        size - nh
    ) // 2

    canvas[
        top:top+nh,
        left:left+nw
    ] = resized

    return (
        canvas,
        scale,
        left,
        top
    )


# ==================================================================================================
# 14. PREPARE MODEL INPUT
# ==================================================================================================

def prepare_image_input(
    image_bgr,
    layout
):

    rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    x = rgb.astype(
        np.float32
    ) / 255.0

    if layout == "NCHW":

        x = np.transpose(
            x,
            (2, 0, 1)
        )

    return np.expand_dims(
        x,
        axis=0
    )


# ==================================================================================================
# 15. PALM OUTPUT PARSER
# ==================================================================================================

def parse_palm_outputs(result):

    arrays = []

    for output in palm_compiled.outputs:

        arr = np.asarray(
            result[output]
        )

        arrays.append(arr)

    boxes = None
    scores = None

    for arr in arrays:

        flat_last = arr.shape[-1]

        if flat_last == 18:
            boxes = arr.reshape(-1, 18)

        elif flat_last == 1:
            scores = arr.reshape(-1)

    if boxes is None or scores is None:

        # Fallback by total size
        for arr in arrays:

            if arr.size == 2016 * 18:
                boxes = arr.reshape(2016, 18)

            elif arr.size == 2016:
                scores = arr.reshape(2016)

    if boxes is None or scores is None:

        raise RuntimeError(
            "Could not identify palm outputs."
        )

    return boxes, scores


# ==================================================================================================
# 16. PALM DECODER
# ==================================================================================================

def decode_palms(
    raw_boxes,
    raw_scores
):

    scores = sigmoid(
        raw_scores
    )

    indices = np.where(
        scores >= PALM_THRESHOLD
    )[0]

    detections = []

    for i in indices:

        r = raw_boxes[i]
        a = PALM_ANCHORS[i]

        cx = (
            r[0] / PALM_SIZE
            +
            a[0]
        )

        cy = (
            r[1] / PALM_SIZE
            +
            a[1]
        )

        w = r[2] / PALM_SIZE
        h = r[3] / PALM_SIZE

        x1 = cx - w / 2
        y1 = cy - h / 2

        x2 = cx + w / 2
        y2 = cy + h / 2

        keypoints = []

        for k in range(7):

            kx = (
                r[4 + 2*k]
                /
                PALM_SIZE
                +
                a[0]
            )

            ky = (
                r[5 + 2*k]
                /
                PALM_SIZE
                +
                a[1]
            )

            keypoints.append(
                [kx, ky]
            )

        detections.append({
            "score": float(scores[i]),
            "bbox": np.array(
                [x1, y1, x2, y2],
                dtype=np.float32
            ),
            "keypoints": np.asarray(
                keypoints,
                dtype=np.float32
            )
        })

    return detections


# ==================================================================================================
# 17. IOU
# ==================================================================================================

def bbox_iou(a, b):

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)

    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(
        0.0,
        ix2 - ix1
    )

    ih = max(
        0.0,
        iy2 - iy1
    )

    inter = iw * ih

    area_a = max(
        0.0,
        ax2 - ax1
    ) * max(
        0.0,
        ay2 - ay1
    )

    area_b = max(
        0.0,
        bx2 - bx1
    ) * max(
        0.0,
        by2 - by1
    )

    union = (
        area_a +
        area_b -
        inter
    )

    if union <= 0:
        return 0.0

    return inter / union


# ==================================================================================================
# 18. WEIGHTED NMS
# ==================================================================================================

def weighted_nms(
    detections,
    threshold=NMS_THRESHOLD,
    max_hands=MAX_HANDS
):

    if not detections:
        return []

    remaining = sorted(
        detections,
        key=lambda d: d["score"],
        reverse=True
    )

    output = []

    while remaining and len(output) < max_hands:

        first = remaining[0]

        overlapping = []
        rest = []

        for d in remaining:

            if bbox_iou(
                first["bbox"],
                d["bbox"]
            ) > threshold:

                overlapping.append(d)

            else:
                rest.append(d)

        weights = np.asarray(
            [
                d["score"]
                for d in overlapping
            ],
            dtype=np.float32
        )

        weights = weights / (
            np.sum(weights) + 1e-8
        )

        bbox = np.sum(
            np.stack(
                [
                    d["bbox"]
                    for d in overlapping
                ]
            )
            *
            weights[:, None],
            axis=0
        )

        kps = np.sum(
            np.stack(
                [
                    d["keypoints"]
                    for d in overlapping
                ]
            )
            *
            weights[:, None, None],
            axis=0
        )

        score = max(
            d["score"]
            for d in overlapping
        )

        output.append({
            "score": float(score),
            "bbox": bbox.astype(np.float32),
            "keypoints": kps.astype(np.float32)
        })

        remaining = rest

    return output


# ==================================================================================================
# 19. REMOVE LETTERBOX
# ==================================================================================================

def undo_letterbox_detection(
    detection,
    frame_width,
    frame_height,
    scale,
    left,
    top
):

    def convert_point(x, y):

        px = (
            x * PALM_SIZE
            -
            left
        ) / scale

        py = (
            y * PALM_SIZE
            -
            top
        ) / scale

        return np.array(
            [px, py],
            dtype=np.float32
        )

    bbox = detection["bbox"]

    p1 = convert_point(
        bbox[0],
        bbox[1]
    )

    p2 = convert_point(
        bbox[2],
        bbox[3]
    )

    keypoints = np.stack([
        convert_point(
            p[0],
            p[1]
        )
        for p in detection["keypoints"]
    ])

    return {
        "score": detection["score"],
        "bbox": np.array(
            [
                p1[0],
                p1[1],
                p2[0],
                p2[1]
            ],
            dtype=np.float32
        ),
        "keypoints": keypoints
    }


# ==================================================================================================
# 20. NORMALIZE ANGLE
# ==================================================================================================

def normalize_angle(angle):

    return (
        angle + math.pi
    ) % (
        2 * math.pi
    ) - math.pi


# ==================================================================================================
# 21. PALM -> HAND ROI
# ==================================================================================================

def palm_to_roi(
    detection
):

    bbox = detection["bbox"]
    keypoints = detection["keypoints"]

    x1, y1, x2, y2 = bbox

    rect_w = max(
        x2 - x1,
        1.0
    )

    rect_h = max(
        y2 - y1,
        1.0
    )

    cx = (
        x1 + x2
    ) / 2.0

    cy = (
        y1 + y2
    ) / 2.0

    wrist = keypoints[0]

    # MediaPipe palm keypoint 2:
    # middle finger MCP region.
    middle = keypoints[2]

    rotation = normalize_angle(
        math.pi / 2
        -
        math.atan2(
            -(middle[1] - wrist[1]),
            middle[0] - wrist[0]
        )
    )

    shift_y = -0.5

    x_shift = (
        -rect_h
        *
        shift_y
        *
        math.sin(rotation)
    )

    y_shift = (
        rect_h
        *
        shift_y
        *
        math.cos(rotation)
    )

    cx += x_shift
    cy += y_shift

    size = max(
        rect_w,
        rect_h
    ) * 2.6

    return {
        "cx": float(cx),
        "cy": float(cy),
        "width": float(size),
        "height": float(size),
        "rotation": float(rotation)
    }


# ==================================================================================================
# 22. ROI CORNERS
# ==================================================================================================

def roi_corners(rect):

    cx = rect["cx"]
    cy = rect["cy"]

    w = rect["width"]
    h = rect["height"]

    theta = rect["rotation"]

    local = np.array([
        [-w/2, -h/2],
        [ w/2, -h/2],
        [ w/2,  h/2],
        [-w/2,  h/2]
    ], dtype=np.float32)

    c = math.cos(theta)
    s = math.sin(theta)

    R = np.array([
        [c, -s],
        [s,  c]
    ], dtype=np.float32)

    pts = (
        local @ R.T
    )

    pts[:, 0] += cx
    pts[:, 1] += cy

    return pts.astype(
        np.float32
    )


# ==================================================================================================
# 23. WARP HAND ROI
# ==================================================================================================

def warp_hand(
    frame,
    rect
):

    src = roi_corners(
        rect
    )

    dst = np.array([
        [0, 0],
        [LANDMARK_SIZE - 1, 0],
        [LANDMARK_SIZE - 1, LANDMARK_SIZE - 1],
        [0, LANDMARK_SIZE - 1]
    ], dtype=np.float32)

    M = cv2.getPerspectiveTransform(
        src,
        dst
    )

    M_inv = cv2.getPerspectiveTransform(
        dst,
        src
    )

    crop = cv2.warpPerspective(
        frame,
        M,
        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)
    )

    return crop, M, M_inv, src


# ==================================================================================================
# 24. PARSE LANDMARK OUTPUT
# ==================================================================================================

def parse_landmark_output(
    result
):

    arrays = []

    for output in landmark_compiled.outputs:

        arr = np.asarray(
            result[output]
        )

        arrays.append(arr)

    landmark_arrays = [
        arr
        for arr in arrays
        if arr.size == 63
    ]

    if not landmark_arrays:

        raise RuntimeError(
            "No 63-value hand landmark output found."
        )

    # The screen-space landmark output generally contains
    # pixel-scale XY values, unlike world landmarks.
    chosen = None

    for arr in landmark_arrays:

        test = arr.reshape(21, 3)

        if np.max(
            np.abs(test[:, :2])
        ) > 2.0:

            chosen = test
            break

    if chosen is None:
        chosen = landmark_arrays[0].reshape(
            21,
            3
        )

    scalars = [
        float(arr.reshape(-1)[0])
        for arr in arrays
        if arr.size == 1
    ]

    return (
        chosen.astype(np.float32),
        scalars
    )


# ==================================================================================================
# 25. INVERSE LANDMARK TRANSFORM
# ==================================================================================================

def landmarks_to_frame(
    landmarks,
    M_inv,
    rect
):

    xy = landmarks[
        :,
        :2
    ].astype(
        np.float32
    )

    projected = cv2.perspectiveTransform(
        xy.reshape(-1, 1, 2),
        M_inv
    ).reshape(-1, 2)

    z = (
        landmarks[:, 2]
        /
        LANDMARK_SIZE
        *
        rect["width"]
    )

    output = np.column_stack([
        projected,
        z
    ])

    return output.astype(
        np.float32
    )


# ==================================================================================================
# 26. COMPLETE HAND DETECTOR
# ==================================================================================================

def detect_hands(
    frame
):

    frame_h, frame_w = frame.shape[:2]

    # ----------------------------------------------------------------------------------------------
    # Palm
    # ----------------------------------------------------------------------------------------------

    palm_start = time.perf_counter()

    lb, scale, left, top = letterbox(
        frame,
        PALM_SIZE
    )

    palm_x = prepare_image_input(
        lb,
        PALM_LAYOUT
    )

    palm_result = palm_compiled(
        [palm_x]
    )

    raw_boxes, raw_scores = parse_palm_outputs(
        palm_result
    )

    detections = decode_palms(
        raw_boxes,
        raw_scores
    )

    detections = weighted_nms(
        detections
    )

    detections = [
        undo_letterbox_detection(
            d,
            frame_w,
            frame_h,
            scale,
            left,
            top
        )
        for d in detections
    ]

    palm_ms = (
        time.perf_counter()
        -
        palm_start
    ) * 1000.0


    # ----------------------------------------------------------------------------------------------
    # Landmark
    # ----------------------------------------------------------------------------------------------

    landmark_start = time.perf_counter()

    hands = []

    for detection in detections:

        rect = palm_to_roi(
            detection
        )

        crop, M, M_inv, corners = warp_hand(
            frame,
            rect
        )

        hand_x = prepare_image_input(
            crop,
            LANDMARK_LAYOUT
        )

        hand_result = landmark_compiled(
            [hand_x]
        )

        raw_landmarks, scalar_outputs = parse_landmark_output(
            hand_result
        )

        frame_landmarks = landmarks_to_frame(
            raw_landmarks,
            M_inv,
            rect
        )

        hands.append({
            "landmarks": frame_landmarks,
            "rect": rect,
            "corners": corners,
            "palm_score": detection["score"],
            "diagnostic_scalars": scalar_outputs
        })

    landmark_ms = (
        time.perf_counter()
        -
        landmark_start
    ) * 1000.0

    return (
        hands,
        palm_ms,
        landmark_ms
    )


# ==================================================================================================
# 27. DRAW HAND
# ==================================================================================================

HAND_CONNECTIONS = [
    (0,1),
    (1,2),
    (2,3),
    (3,4),

    (0,5),
    (5,6),
    (6,7),
    (7,8),

    (5,9),
    (9,10),
    (10,11),
    (11,12),

    (9,13),
    (13,14),
    (14,15),
    (15,16),

    (13,17),
    (17,18),
    (18,19),
    (19,20),

    (0,17)
]


def draw_hand(
    frame,
    result
):

    landmarks = result["landmarks"]

    corners = result.get(
        "corners",
        None
    )

    # ROI
    if corners is not None:

        poly = np.round(
            corners
        ).astype(
            np.int32
        )

        cv2.polylines(
            frame,
            [poly],
            True,
            (255, 255, 0),
            1,
            cv2.LINE_AA
        )

    # Skeleton
    for a, b in HAND_CONNECTIONS:

        pa = tuple(
            np.round(
                landmarks[a, :2]
            ).astype(int)
        )

        pb = tuple(
            np.round(
                landmarks[b, :2]
            ).astype(int)
        )

        cv2.line(
            frame,
            pa,
            pb,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    # Landmarks
    for i, p in enumerate(landmarks):

        pt = tuple(
            np.round(
                p[:2]
            ).astype(int)
        )

        radius = (
            5
            if i == 0
            else 3
        )

        cv2.circle(
            frame,
            pt,
            radius,
            (0, 0, 255),
            -1,
            cv2.LINE_AA
        )


# ==================================================================================================
# 28. LANDMARK -> L26C FEATURE
# ==================================================================================================

def normalize_for_classifier(
    landmarks
):

    xyz = np.asarray(
        landmarks,
        dtype=np.float32
    ).reshape(
        21,
        3
    )

    if not np.all(
        np.isfinite(xyz)
    ):
        return None

    # Wrist-centered
    xyz = (
        xyz
        -
        xyz[0:1]
    )

    # Same normalization used by L26C:
    # wrist -> middle MCP (landmark 9)
    scale = float(
        np.linalg.norm(
            xyz[9]
        )
    )

    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):
        return None

    xyz = (
        xyz /
        scale
    )

    feature = xyz.reshape(
        FEATURE_DIM
    ).astype(
        np.float32
    )

    return feature


# ==================================================================================================
# 29. SOFTMAX
# ==================================================================================================

def softmax(
    x
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    x = (
        x -
        np.max(x)
    )

    e = np.exp(x)

    return (
        e /
        (
            np.sum(e)
            +
            1e-12
        )
    )


# ==================================================================================================
# 30. CLASSIFY 32-FRAME WINDOW
# ==================================================================================================

def classify_sequence(
    sequence
):

    sequence = np.asarray(
        sequence,
        dtype=np.float32
    )

    if sequence.shape != (
        SEQUENCE_LENGTH,
        FEATURE_DIM
    ):

        raise ValueError(
            f"Expected {(SEQUENCE_LENGTH, FEATURE_DIM)}, "
            f"got {sequence.shape}"
        )

    x = sequence[
        None,
        ...
    ]

    t0 = time.perf_counter()

    result = classifier_compiled(
        [x]
    )

    inference_ms = (
        time.perf_counter()
        -
        t0
    ) * 1000.0

    logits = np.asarray(
        result[classifier_output]
    ).reshape(-1)

    probs = softmax(
        logits
    )

    order = np.argsort(
        probs
    )[::-1]

    top1 = int(
        order[0]
    )

    top2 = int(
        order[1]
    )

    confidence = float(
        probs[top1]
    )

    margin = float(
        probs[top1]
        -
        probs[top2]
    )

    # normalized entropy: 0 confident, 1 uniform
    entropy = float(
        -np.sum(
            probs
            *
            np.log(
                probs + 1e-12
            )
        )
        /
        np.log(
            len(probs)
        )
    )

    return {
        "prediction": CLASS_NAMES[top1],
        "class_id": top1,
        "confidence": confidence,
        "margin": margin,
        "entropy": entropy,
        "probabilities": probs,
        "inference_ms": inference_ms
    }


# ==================================================================================================
# 31. L27 — NO_SIGN REJECTION ONLY
# ==================================================================================================

def l27_reject(
    metrics,
    hand_ratio
):

    confidence = float(
        metrics["confidence"]
    )

    margin = float(
        metrics["margin"]
    )

    entropy = float(
        metrics["entropy"]
    )

    # IMPORTANT:
    #
    # This function returns ONLY:
    #
    # NO_SIGN
    # or
    # SIGN
    #
    # It never returns another class.

    if hand_ratio < HAND_RATIO_THRESHOLD:
        return "NO_SIGN"

    if confidence < CONF_THRESHOLD:
        return "NO_SIGN"

    if margin < MARGIN_THRESHOLD:
        return "NO_SIGN"

    if entropy > ENTROPY_THRESHOLD:
        return "NO_SIGN"

    return "SIGN"


# ==================================================================================================
# 32. ASYNC TTS
# ==================================================================================================

class AsyncTTS:

    def __init__(
        self,
        rate=150,
        volume=1.0
    ):

        self.rate = rate
        self.volume = volume

        self.q = queue.Queue()

        self.running = True
        self.muted = False

        self.spoken_count = 0

        self.thread = threading.Thread(
            target=self._worker,
            daemon=True
        )

        self.thread.start()


    def _worker(self):

        try:

            engine = pyttsx3.init()

            engine.setProperty(
                "rate",
                self.rate
            )

            engine.setProperty(
                "volume",
                self.volume
            )

        except Exception as e:

            print(
                "[TTS ERROR] Initialization failed:",
                e
            )

            return

        while self.running:

            try:

                text = self.q.get(
                    timeout=0.2
                )

            except queue.Empty:

                continue

            if text is None:

                self.q.task_done()
                break

            if not self.muted:

                try:

                    engine.say(
                        str(text)
                    )

                    engine.runAndWait()

                    self.spoken_count += 1

                except Exception as e:

                    print(
                        "[TTS ERROR]",
                        e
                    )

            self.q.task_done()

        try:
            engine.stop()
        except Exception:
            pass


    def speak(
        self,
        text
    ):

        text = str(
            text
        ).strip()

        if (
            text
            and
            not self.muted
        ):

            self.q.put(
                text
            )


    def clear_queue(
        self
    ):

        try:

            while True:

                self.q.get_nowait()
                self.q.task_done()

        except queue.Empty:

            pass


    def toggle_mute(
        self
    ):

        self.muted = (
            not self.muted
        )

        if self.muted:
            self.clear_queue()

        return self.muted


    def stop(
        self
    ):

        self.running = False

        self.clear_queue()

        try:
            self.q.put_nowait(None)
        except Exception:
            pass


# ==================================================================================================
# 33. START TTS
# ==================================================================================================

tts = AsyncTTS(
    rate=TTS_RATE,
    volume=TTS_VOLUME
)

print("[PASS] TTS background thread started.")


# ==================================================================================================
# 34. TEMPORAL STATE
# ==================================================================================================

feature_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)

hand_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)

model_history = deque(
    maxlen=MODEL_HISTORY_SIZE
)

last_feature = None


# ==================================================================================================
# 35. SENTENCE STATE
# ==================================================================================================

sentence_tokens = []

token_events = []

speech_events = []

active_token = None

nosign_count = 0


# ==================================================================================================
# 36. DISPLAY STATE
# ==================================================================================================

model_prediction = "-"

output_prediction = "NO_SIGN"

confidence = 0.0
margin = 0.0
entropy = 1.0

classifier_ms = 0.0

hand_ratio = 0.0

last_spoken_text = "-"


# ==================================================================================================
# 37. SESSION METRICS
# ==================================================================================================

frame_count = 0
hand_frames = 0

classification_windows = 0

valid_windows = 0
nosign_windows = 0

tokens_added = 0

session_start = time.perf_counter()

fps_start = time.perf_counter()
fps_frames = 0

live_fps = 0.0


# ==================================================================================================
# 38. MODEL-DIRECT TOKEN CONFIRMATION
# ==================================================================================================

def get_confirmed_model_prediction():

    # IMPORTANT:
    #
    # This votes ONLY over actual L26C model predictions.
    #
    # It cannot choose a class that was not predicted
    # by the trained model.

    if len(
        model_history
    ) < MODEL_CONFIRM_VOTES:

        return None

    counts = Counter(
        model_history
    )

    candidate, votes = (
        counts.most_common(1)[0]
    )

    if votes >= MODEL_CONFIRM_VOTES:
        return candidate

    return None


# ==================================================================================================
# 39. SPEAK HELPER
# ==================================================================================================

def speak_text(
    text,
    source
):

    global last_spoken_text

    text = str(
        text
    ).strip()

    if not text:
        return

    if tts.muted:
        return

    tts.speak(
        text
    )

    last_spoken_text = text

    speech_events.append({
        "time_seconds":
            float(
                time.perf_counter()
                -
                session_start
            ),

        "source":
            source,

        "text":
            text
    })

    print(
        "[TTS]",
        text
    )


# ==================================================================================================
# 40. OPEN CAMERA
# ==================================================================================================

cap = cv2.VideoCapture(
    CAMERA_ID
)

if not cap.isOpened():

    tts.stop()

    raise RuntimeError(
        "Could not open webcam."
    )

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    CAMERA_WIDTH
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    CAMERA_HEIGHT
)

try:

    cap.set(
        cv2.CAP_PROP_BUFFERSIZE,
        1
    )

except Exception:
    pass


print()
print("=" * 100)
print("L30 — FULL STANDALONE REAL-TIME SIGN LANGUAGE + TTS")
print("=" * 100)

print()
print("Sign identity : L26C TRAINED MODEL")
print("L27 role      : NO_SIGN ONLY")
print("L28 override  : DISABLED")
print("TTS           : ASYNCHRONOUS")
print()

print("Vocabulary:")
print(CLASS_NAMES)

print()
print("Q/ESC : quit")
print("C     : clear sequence")
print("R     : reset recognition")
print("B     : delete last token")
print("S     : speak full sequence")
print("M     : mute/unmute")
print()


# ==================================================================================================
# 41. LIVE LOOP
# ==================================================================================================

try:

    while True:

        ok, frame = cap.read()

        if not ok:
            continue

        frame_count += 1


        # ==========================================================================================
        # HAND DETECTION
        # ==========================================================================================

        try:

            results, palm_ms, landmark_ms = detect_hands(
                frame
            )

        except Exception as e:

            results = []

            palm_ms = 0.0
            landmark_ms = 0.0


        current_feature = None
        hand_present = False


        # ==========================================================================================
        # CURRENT HAND FEATURE
        # ==========================================================================================

        if results:

            current_feature = normalize_for_classifier(
                results[0]["landmarks"]
            )

            if current_feature is not None:

                hand_present = True

                hand_frames += 1

                last_feature = (
                    current_feature.copy()
                )


        # ==========================================================================================
        # HAND BUFFER
        # ==========================================================================================

        hand_buffer.append(
            int(
                hand_present
            )
        )

        hand_ratio = (
            float(
                np.mean(
                    hand_buffer
                )
            )
            if hand_buffer
            else 0.0
        )


        # ==========================================================================================
        # TEMPORAL FEATURE BUFFER
        # ==========================================================================================

        if current_feature is not None:

            feature_buffer.append(
                current_feature.copy()
            )

        elif last_feature is not None:

            # Keep temporal length continuous.
            #
            # L27 hand evidence is still responsible
            # for rejecting prolonged missing-hand periods.

            feature_buffer.append(
                last_feature.copy()
            )


        # ==========================================================================================
        # DEFAULT OUTPUT
        # ==========================================================================================

        output_prediction = "NO_SIGN"


        # ==========================================================================================
        # L26C MODEL
        # ==========================================================================================

        if len(feature_buffer) == SEQUENCE_LENGTH:

            metrics = classify_sequence(
                np.stack(
                    feature_buffer,
                    axis=0
                )
            )

            classification_windows += 1


            # --------------------------------------------------------------------------------------
            # AUTHORITATIVE SIGN IDENTITY
            # --------------------------------------------------------------------------------------

            model_prediction = (
                metrics["prediction"]
            )

            confidence = float(
                metrics["confidence"]
            )

            margin = float(
                metrics["margin"]
            )

            entropy = float(
                metrics["entropy"]
            )

            classifier_ms = float(
                metrics["inference_ms"]
            )


            # ======================================================================================
            # L27 — ONLY ASK "NO_SIGN OR SIGN?"
            # ======================================================================================

            rejection_result = l27_reject(
                metrics,
                hand_ratio
            )

            is_nosign = (
                rejection_result
                ==
                "NO_SIGN"
            )


            # ======================================================================================
            # NO_SIGN
            # ======================================================================================

            if is_nosign:

                output_prediction = (
                    "NO_SIGN"
                )

                nosign_windows += 1

                nosign_count += 1

                # Critical:
                # stale model labels must not leak into next token.

                model_history.clear()


                # Release active sign after enough NO_SIGN evidence.

                if (
                    nosign_count
                    >=
                    NO_SIGN_RELEASE_FRAMES
                ):

                    active_token = None


            # ======================================================================================
            # VALID SIGN
            # ======================================================================================

            else:

                valid_windows += 1

                nosign_count = 0


                # ==================================================================================
                # CRITICAL RULE
                #
                # OUTPUT = MODEL
                #
                # No stable label is allowed to replace this.
                # ==================================================================================

                output_prediction = (
                    model_prediction
                )


                # Only actual model predictions enter token voting.

                model_history.append(
                    model_prediction
                )


                confirmed_sign = (
                    get_confirmed_model_prediction()
                )


                # ==================================================================================
                # NEW TOKEN EVENT
                # ==================================================================================

                if confirmed_sign is not None:

                    # New token only after previous token
                    # has been released through NO_SIGN.

                    if active_token is None:

                        active_token = (
                            confirmed_sign
                        )

                        sentence_tokens.append(
                            confirmed_sign
                        )

                        if (
                            len(sentence_tokens)
                            >
                            MAX_SENTENCE_TOKENS
                        ):

                            sentence_tokens = (
                                sentence_tokens[
                                    -MAX_SENTENCE_TOKENS:
                                ]
                            )

                        tokens_added += 1

                        event = {

                            "event_index":
                                int(
                                    tokens_added
                                ),

                            "time_seconds":
                                float(
                                    time.perf_counter()
                                    -
                                    session_start
                                ),

                            "frame":
                                int(
                                    frame_count
                                ),

                            "token":
                                confirmed_sign,

                            "model_prediction":
                                model_prediction,

                            "confidence":
                                confidence,

                            "margin":
                                margin,

                            "entropy":
                                entropy,

                            "hand_ratio":
                                hand_ratio,

                            "sentence":
                                " ".join(
                                    sentence_tokens
                                )
                        }

                        token_events.append(
                            event
                        )

                        print()
                        print(
                            "[MODEL TOKEN]",
                            confirmed_sign.upper()
                        )

                        print(
                            "[SEQUENCE]",
                            " ".join(
                                sentence_tokens
                            )
                        )


                        # =============================================================================
                        # L30 — SPEAK NEW TOKEN
                        # =============================================================================

                        if AUTO_SPEAK_NEW_TOKEN:

                            speak_text(
                                confirmed_sign,
                                source="new_model_token"
                            )


                    # Same sign being held:
                    # don't append and don't speak again.

                    elif (
                        confirmed_sign
                        ==
                        active_token
                    ):

                        pass


                    # Different prediction before NO_SIGN release:
                    #
                    # Do NOT immediately append it.
                    # Wait for separation between sign events.

                    else:

                        pass


        # ==========================================================================================
        # DRAW
        # ==========================================================================================

        display = frame.copy()

        for result in results:

            try:

                draw_hand(
                    display,
                    result
                )

            except Exception:
                pass


        # ==========================================================================================
        # FPS
        # ==========================================================================================

        fps_frames += 1

        now = time.perf_counter()

        elapsed = (
            now -
            fps_start
        )

        if elapsed >= 0.5:

            live_fps = (
                fps_frames /
                elapsed
            )

            fps_frames = 0
            fps_start = now


        # ==========================================================================================
        # UI BACKGROUND
        # ==========================================================================================

        overlay = display.copy()

        cv2.rectangle(
            overlay,
            (0, 0),
            (
                display.shape[1],
                345
            ),
            (0, 0, 0),
            -1
        )

        cv2.addWeighted(
            overlay,
            0.70,
            display,
            0.30,
            0,
            display
        )


        # ==========================================================================================
        # COLORS
        # ==========================================================================================

        if output_prediction == "NO_SIGN":

            output_color = (
                0,
                165,
                255
            )

        else:

            output_color = (
                0,
                255,
                0
            )


        # ==========================================================================================
        # HEADER
        # ==========================================================================================

        cv2.putText(
            display,
            f"L30 | FPS {live_fps:.1f}",
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # MODEL PREDICTION
        # ==========================================================================================

        cv2.putText(
            display,
            "MODEL: " + model_prediction.upper(),
            (10, 62),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.82,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # OUTPUT
        # ==========================================================================================

        cv2.putText(
            display,
            "OUTPUT: " + output_prediction.upper(),
            (10, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.82,
            output_color,
            2,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # ACTIVE TOKEN
        # ==========================================================================================

        active_text = (
            active_token.upper()
            if active_token is not None
            else "NONE"
        )

        cv2.putText(
            display,
            "ACTIVE TOKEN: " + active_text,
            (10, 130),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.43,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # METRICS
        # ==========================================================================================

        cv2.putText(
            display,
            (
                f"Conf {confidence:.3f} | "
                f"Margin {margin:.3f} | "
                f"Entropy {entropy:.3f}"
            ),
            (10, 157),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.38,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

        cv2.putText(
            display,
            (
                f"Hand {hand_ratio:.2f} | "
                f"NO_SIGN {nosign_count}/{NO_SIGN_RELEASE_FRAMES} | "
                f"GRU {classifier_ms:.2f} ms"
            ),
            (10, 183),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.36,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # MODEL VOTES
        # ==========================================================================================

        votes_text = (
            "Votes: "
            +
            ",".join(
                list(
                    model_history
                )
            )
        )

        cv2.putText(
            display,
            votes_text[-90:],
            (10, 209),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.34,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # TTS
        # ==========================================================================================

        tts_status = (
            "MUTED"
            if tts.muted
            else "ON"
        )

        cv2.putText(
            display,
            (
                f"TTS: {tts_status} | "
                f"LAST: {last_spoken_text.upper()}"
            ),
            (10, 235),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.39,
            (0, 255, 255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # SEQUENCE
        # ==========================================================================================

        sentence = " ".join(
            sentence_tokens
        )

        if not sentence:

            sentence = (
                "<waiting for sign>"
            )

        display_sentence = (
            sentence[-75:]
        )

        cv2.putText(
            display,
            "SEQUENCE:",
            (10, 266),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.43,
            (255, 255, 0),
            1,
            cv2.LINE_AA
        )

        cv2.putText(
            display,
            display_sentence,
            (10, 293),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.51,
            (0, 255, 255),
            1,
            cv2.LINE_AA
        )

        cv2.putText(
            display,
            "S speak | M mute | C clear | R reset | B delete | Q quit",
            (10, 326),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.32,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # WINDOW
        # ==========================================================================================

        cv2.imshow(
            "CISLR L30 - Sign Language To Speech",
            display
        )


        # ==========================================================================================
        # KEY
        # ==========================================================================================

        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        # ------------------------------------------------------------------------------------------
        # QUIT
        # ------------------------------------------------------------------------------------------

        if (
            key == ord("q")
            or
            key == 27
        ):

            break


        # ------------------------------------------------------------------------------------------
        # SPEAK FULL SEQUENCE
        # ------------------------------------------------------------------------------------------

        elif key == ord("s"):

            full_text = " ".join(
                sentence_tokens
            )

            if full_text:

                speak_text(
                    full_text,
                    source="manual_full_sequence"
                )

                print(
                    "[SPEAK FULL]",
                    full_text
                )

            else:

                print(
                    "[TTS] No sequence to speak."
                )


        # ------------------------------------------------------------------------------------------
        # MUTE
        # ------------------------------------------------------------------------------------------

        elif key == ord("m"):

            muted = (
                tts.toggle_mute()
            )

            print(
                "[TTS]",
                "MUTED"
                if muted
                else "UNMUTED"
            )


        # ------------------------------------------------------------------------------------------
        # CLEAR EVERYTHING INCLUDING SENTENCE
        # ------------------------------------------------------------------------------------------

        elif key == ord("c"):

            sentence_tokens.clear()

            token_events.clear()

            active_token = None

            nosign_count = 0

            model_history.clear()

            feature_buffer.clear()

            hand_buffer.clear()

            last_feature = None

            tts.clear_queue()

            model_prediction = "-"
            output_prediction = "NO_SIGN"

            print(
                "[CLEAR] Sequence cleared."
            )


        # ------------------------------------------------------------------------------------------
        # RESET RECOGNITION, KEEP SENTENCE
        # ------------------------------------------------------------------------------------------

        elif key == ord("r"):

            active_token = None

            nosign_count = 0

            model_history.clear()

            feature_buffer.clear()

            hand_buffer.clear()

            last_feature = None

            model_prediction = "-"
            output_prediction = "NO_SIGN"

            confidence = 0.0
            margin = 0.0
            entropy = 1.0

            print(
                "[RESET] Recognition reset."
            )


        # ------------------------------------------------------------------------------------------
        # DELETE LAST TOKEN
        # ------------------------------------------------------------------------------------------

        elif key == ord("b"):

            if sentence_tokens:

                removed = (
                    sentence_tokens.pop()
                )

                print(
                    "[DELETE]",
                    removed
                )

                print(
                    "[SEQUENCE]",
                    " ".join(
                        sentence_tokens
                    )
                )


# ==================================================================================================
# 42. CLEANUP
# ==================================================================================================

finally:

    cap.release()

    cv2.destroyAllWindows()

    tts.stop()


# ==================================================================================================
# 43. SESSION RESULTS
# ==================================================================================================

duration = (
    time.perf_counter()
    -
    session_start
)

overall_fps = (
    frame_count / duration
    if duration > 0
    else 0.0
)

hand_detection_rate = (
    hand_frames / frame_count
    if frame_count > 0
    else 0.0
)

final_sequence = " ".join(
    sentence_tokens
)


# ==================================================================================================
# 44. SAVE TOKEN EVENTS
# ==================================================================================================

TOKEN_EVENT_PATH = (
    REPORT_ROOT /
    "l30_token_events.json"
)

with open(
    TOKEN_EVENT_PATH,
    "w"
) as f:

    json.dump(
        token_events,
        f,
        indent=2
    )


# ==================================================================================================
# 45. SAVE SPEECH EVENTS
# ==================================================================================================

SPEECH_EVENT_PATH = (
    REPORT_ROOT /
    "l30_speech_events.json"
)

with open(
    SPEECH_EVENT_PATH,
    "w"
) as f:

    json.dump(
        speech_events,
        f,
        indent=2
    )


# ==================================================================================================
# 46. SAVE FINAL TEXT
# ==================================================================================================

TEXT_PATH = (
    REPORT_ROOT /
    "l30_final_text.txt"
)

with open(
    TEXT_PATH,
    "w"
) as f:

    f.write(
        final_sequence
    )


# ==================================================================================================
# 47. SAVE CONFIG
# ==================================================================================================

CONFIG_PATH = (
    CONFIG_ROOT /
    "l30_config.json"
)

config = {

    "stage":
        "L30",

    "device":
        DEVICE,

    "sign_identity_source":
        "L26C trained OpenVINO BiGRU",

    "l27_role":
        "NO_SIGN rejection only",

    "l28_class_override":
        False,

    "sequence_length":
        SEQUENCE_LENGTH,

    "feature_dim":
        FEATURE_DIM,

    "model_history_size":
        MODEL_HISTORY_SIZE,

    "model_confirm_votes":
        MODEL_CONFIRM_VOTES,

    "nosign_release_frames":
        NO_SIGN_RELEASE_FRAMES,

    "tts_engine":
        "pyttsx3",

    "tts_rate":
        TTS_RATE,

    "tts_volume":
        TTS_VOLUME,

    "tts_async":
        True,

    "auto_speak_new_token":
        AUTO_SPEAK_NEW_TOKEN,

    "vocabulary":
        CLASS_NAMES,

    "locked_test_used":
        False,

    "training":
        False,

    "retraining":
        False
}

with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ==================================================================================================
# 48. SUMMARY
# ==================================================================================================

SUMMARY_PATH = (
    REPORT_ROOT /
    "l30_summary.json"
)

summary = {

    "stage":
        "L30",

    "status":
        "DEVELOPMENT_RUN_COMPLETE",

    "frames":
        int(
            frame_count
        ),

    "duration_seconds":
        float(
            duration
        ),

    "overall_fps":
        float(
            overall_fps
        ),

    "hand_detection_rate":
        float(
            hand_detection_rate
        ),

    "classification_windows":
        int(
            classification_windows
        ),

    "valid_sign_windows":
        int(
            valid_windows
        ),

    "nosign_windows":
        int(
            nosign_windows
        ),

    "tokens_generated":
        int(
            tokens_added
        ),

    "speech_requests":
        int(
            len(
                speech_events
            )
        ),

    "final_tokens":
        list(
            sentence_tokens
        ),

    "final_sequence":
        final_sequence,

    "sign_identity_source":
        "L26C_MODEL",

    "nosign_source":
        "L27",

    "locked_test_used":
        False
}

with open(
    SUMMARY_PATH,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ==================================================================================================
# 49. MASTER
# ==================================================================================================

MASTER_PATH = (
    L30_ROOT /
    "L30_TEXT_TO_SPEECH_MASTER.json"
)

master = {

    "stage":
        "L30",

    "status":
        "DEVELOPMENT_RUN_COMPLETE",

    "pipeline": (
        "webcam -> OpenVINO palm -> OpenVINO landmarks -> "
        "L26C OpenVINO BiGRU -> L27 NO_SIGN gate -> "
        "model-direct token sequence -> asynchronous TTS"
    ),

    "critical_rule": (
        "L26C model is authoritative for sign identity. "
        "L27 may only reject the prediction as NO_SIGN and "
        "cannot substitute another sign class."
    ),

    "artifacts": {

        "token_events":
            str(
                TOKEN_EVENT_PATH
            ),

        "speech_events":
            str(
                SPEECH_EVENT_PATH
            ),

        "final_text":
            str(
                TEXT_PATH
            ),

        "config":
            str(
                CONFIG_PATH
            ),

        "summary":
            str(
                SUMMARY_PATH
            )
    },

    "integrity": {

        "locked_test_used":
            False,

        "training":
            False,

        "retraining":
            False,

        "videomae":
            False
    }
}

with open(
    MASTER_PATH,
    "w"
) as f:

    json.dump(
        master,
        f,
        indent=2
    )


# ==================================================================================================
# 50. MASTER HASH
# ==================================================================================================

with open(
    MASTER_PATH,
    "rb"
) as f:

    master_sha = hashlib.sha256(
        f.read()
    ).hexdigest()


# ==================================================================================================
# 51. FINAL REPORT
# ==================================================================================================

print()
print("=" * 100)
print("L30 — SESSION COMPLETE")
print("=" * 100)

print()
print(
    "Frames                 :",
    frame_count
)

print(
    "Duration               :",
    f"{duration:.2f} sec"
)

print(
    "Overall FPS            :",
    f"{overall_fps:.2f}"
)

print(
    "Hand detection rate    :",
    f"{hand_detection_rate * 100:.2f}%"
)

print()
print(
    "Classification windows :",
    classification_windows
)

print(
    "Valid sign windows     :",
    valid_windows
)

print(
    "NO_SIGN windows        :",
    nosign_windows
)

print()
print(
    "Tokens generated       :",
    tokens_added
)

print(
    "Speech requests        :",
    len(
        speech_events
    )
)

print()
print("FINAL TOKENS:")
print(
    sentence_tokens
)

print()
print("FINAL SEQUENCE:")
print(
    final_sequence
)

print()
print("-" * 100)

print(
    "Sign identity          : L26C TRAINED MODEL"
)

print(
    "NO_SIGN                : L27"
)

print(
    "L28 sign override      : DISABLED"
)

print(
    "TTS                    : pyttsx3 ASYNC"
)

print(
    "Locked test used       : NO"
)

print(
    "Training/retraining    : NO"
)

print()
print(
    "Master:"
)

print(
    MASTER_PATH
)

print()
print(
    "Master SHA256:"
)

print(
    master_sha
)

print("-" * 100)

print()
print(
    "[PASS] L30 standalone development run complete."
)

[PASS] Required model/config files found.
[INFO] L27 config loaded:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l27_nosign_rejection/config/l27_rejection_thresholds.json

L27 thresholds
----------------------------
Confidence >= 0.1
Margin     >= 0.0
Entropy    <= 0.8
Hand ratio >= 0.8

[INFO] OpenVINO: 2026.3.1-22476-759c5a6ab8c-releases/2026/3
[INFO] Devices: ['CPU', 'GPU']
[PASS] Palm detector loaded.
       input: [1,192,192,3]
[PASS] Hand landmark model loaded.
       input: [1,224,224,3]
[PASS] L26C BiGRU loaded.
       input : [1,32,63]
       output: [1,10]
[INFO] Palm layout    : NHWC
[INFO] Landmark layout: NHWC
[PASS] Palm anchors: (2016, 4)
[PASS] TTS background thread started.

L30 — FULL STANDALONE REAL-TIME SIGN LANGUAGE + TTS

Sign identity : L26C TRAINED MODEL
L27 role      : NO_SIGN ONLY
L28 override  : DISABLED
TTS           : ASYNCHRONOUS

Vocabulary:
['monday', 'interest', 'thursday', 'christmas', 'pencil', 'mother', 'tuesday', 'friday', 'ru

QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or sw


[MODEL TOKEN] PENCIL
[SEQUENCE] pencil
[TTS] pencil

[MODEL TOKEN] INTEREST
[SEQUENCE] pencil interest
[TTS] interest

[MODEL TOKEN] WASH
[SEQUENCE] pencil interest wash
[TTS] wash

[MODEL TOKEN] INTEREST
[SEQUENCE] pencil interest wash interest
[TTS] interest

[MODEL TOKEN] RUBBER
[SEQUENCE] pencil interest wash interest rubber
[TTS] rubber

[MODEL TOKEN] INTEREST
[SEQUENCE] pencil interest wash interest rubber interest
[TTS] interest

L30 — SESSION COMPLETE

Frames                 : 1689
Duration               : 90.87 sec
Overall FPS            : 18.59
Hand detection rate    : 88.63%

Classification windows : 1658
Valid sign windows     : 1351
NO_SIGN windows        : 307

Tokens generated       : 6
Speech requests        : 6

FINAL TOKENS:
['pencil', 'interest', 'wash', 'interest', 'rubber', 'interest']

FINAL SEQUENCE:
pencil interest wash interest rubber interest

----------------------------------------------------------------------------------------------------
Sign identity   

In [3]:
# ==================================================================================================
# CISLR — L31 + L32 UPDATED
# PROFESSIONAL GUI + CONTINUOUS SIGN TRANSITIONS + END-TO-END EVALUATION
# ==================================================================================================
#
# IMPORTANT:
# Run the working L30 FULL STANDALONE cell first.
# Quit L30 using Q.
# Then run THIS CELL.
#
# PIPELINE
#
# Webcam
#   ↓
# OpenVINO Palm
#   ↓
# OpenVINO Hand Landmarks
#   ↓
# L26C BiGRU
#   ↓
# MODEL chooses sign
#   ↓
# L27 may ONLY reject as NO_SIGN
#   ↓
# 3-of-5 model confirmation
#   ↓
# token transition:
#
#      FRIDAY -> MOTHER
#          ↓
#      friday mother
#
# NO_SIGN is NOT required between different valid signs.
#
# Holding same sign:
#
#      MOTHER MOTHER MOTHER ...
#          ↓
#      one "mother" token only
#
# ==================================================================================================

from pathlib import Path
from collections import deque, Counter
from datetime import datetime

import tkinter as tk
from tkinter import ttk, messagebox

import cv2
import numpy as np

import threading
import queue
import time
import json
import csv
import hashlib
import platform
import os
import sys


# ==================================================================================================
# 1. DEPENDENCIES
# ==================================================================================================

try:
    from PIL import Image, ImageTk
except ImportError:
    raise RuntimeError(
        "Install Pillow:\n"
        "python -m pip install pillow"
    )

try:
    import psutil
except ImportError:
    raise RuntimeError(
        "Install psutil:\n"
        "python -m pip install psutil"
    )

try:
    import pyttsx3
except ImportError:
    raise RuntimeError(
        "Install pyttsx3:\n"
        "python -m pip install pyttsx3"
    )


# ==================================================================================================
# 2. VERIFY L30 RUNTIME
# ==================================================================================================

required_runtime = [
    "detect_hands",
    "draw_hand",
    "normalize_for_classifier",
    "classify_sequence",
    "l27_reject",
    "CLASS_NAMES",
    "SEQUENCE_LENGTH",
    "FEATURE_DIM"
]

missing = [
    name
    for name in required_runtime
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "\nRun your working L30 FULL STANDALONE cell first.\n"
        "Quit L30 with Q and then run L31/L32.\n\n"
        "Missing:\n" +
        "\n".join(missing)
    )

print("[PASS] L30 recognition pipeline detected.")


# ==================================================================================================
# 3. PATHS
# ==================================================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL"
)

L31_ROOT = ROOT / "audit/l31_gui"
L32_ROOT = ROOT / "audit/l32_end_to_end_evaluation"

L31_REPORT_ROOT = L31_ROOT / "reports"
L32_REPORT_ROOT = L32_ROOT / "reports"
L32_CONFIG_ROOT = L32_ROOT / "config"

for p in [
    L31_REPORT_ROOT,
    L32_REPORT_ROOT,
    L32_CONFIG_ROOT
]:
    p.mkdir(parents=True, exist_ok=True)


# ==================================================================================================
# 4. CONFIGURATION
# ==================================================================================================

CAMERA_ID = 0
CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480

MODEL_HISTORY_SIZE = 5
MODEL_CONFIRM_VOTES = 3

# NO_SIGN still clears/releases state, but is NOT mandatory
# between two different valid signs.
NO_SIGN_RELEASE_FRAMES = 3

MAX_SENTENCE_TOKENS = 30

GUI_INTERVAL_MS = 15

RESOURCE_SAMPLE_INTERVAL = 0.5

MAX_METRIC_HISTORY = 100000

TTS_RATE = 150
TTS_VOLUME = 1.0


# ==================================================================================================
# 5. GUI COLORS
# ==================================================================================================

BG = "#0B1120"
PANEL = "#111827"
PANEL_2 = "#172033"
BORDER = "#26334A"

TEXT = "#F8FAFC"
TEXT_SECONDARY = "#94A3B8"

ACCENT = "#38BDF8"
SUCCESS = "#22C55E"
WARNING = "#F59E0B"
DANGER = "#EF4444"
PURPLE = "#A78BFA"


# ==================================================================================================
# 6. TTS
# ==================================================================================================

class GUIAsyncTTS:

    def __init__(self, rate=150, volume=1.0):

        self.rate = rate
        self.volume = volume

        self.q = queue.Queue()

        self.running = True
        self.muted = False

        self.spoken_count = 0
        self.last_spoken = "-"

        self.thread = threading.Thread(
            target=self._worker,
            daemon=True
        )

        self.thread.start()


    def _worker(self):

        try:
            engine = pyttsx3.init()

            engine.setProperty(
                "rate",
                self.rate
            )

            engine.setProperty(
                "volume",
                self.volume
            )

        except Exception as e:
            print("[TTS ERROR]", e)
            return

        while self.running:

            try:
                text = self.q.get(timeout=0.2)

            except queue.Empty:
                continue

            if text is None:
                self.q.task_done()
                break

            if not self.muted:

                try:
                    engine.say(str(text))
                    engine.runAndWait()

                    self.spoken_count += 1
                    self.last_spoken = str(text)

                except Exception as e:
                    print("[TTS ERROR]", e)

            self.q.task_done()

        try:
            engine.stop()
        except Exception:
            pass


    def speak(self, text):

        text = str(text).strip()

        if text and not self.muted:
            self.q.put(text)


    def toggle_mute(self):

        self.muted = not self.muted

        if self.muted:
            self.clear()

        return self.muted


    def clear(self):

        try:
            while True:
                self.q.get_nowait()
                self.q.task_done()

        except queue.Empty:
            pass


    def stop(self):

        self.running = False

        self.clear()

        try:
            self.q.put_nowait(None)
        except Exception:
            pass


# ==================================================================================================
# 7. METRIC FUNCTIONS
# ==================================================================================================

def safe_values(values):

    return [
        float(v)
        for v in values
        if np.isfinite(v)
    ]


def safe_mean(values):

    values = safe_values(values)

    return float(np.mean(values)) if values else 0.0


def safe_median(values):

    values = safe_values(values)

    return float(np.median(values)) if values else 0.0


def safe_percentile(values, p):

    values = safe_values(values)

    return float(np.percentile(values, p)) if values else 0.0


# ==================================================================================================
# 8. MAIN APPLICATION
# ==================================================================================================

class CISLRApplication:

    def __init__(self, root):

        self.root = root

        # ------------------------------------------------------------------------------------------
        # WINDOW
        # ------------------------------------------------------------------------------------------

        self.root.title(
            "CISLR • Real-Time Sign Language Interpreter"
        )

        self.root.geometry("1480x900")
        self.root.minsize(1180, 720)

        self.root.configure(bg=BG)

        self.root.protocol(
            "WM_DELETE_WINDOW",
            self.close
        )


        # ------------------------------------------------------------------------------------------
        # PROCESS
        # ------------------------------------------------------------------------------------------

        self.process = psutil.Process(
            os.getpid()
        )

        self.logical_cpu_count = max(
            psutil.cpu_count(logical=True) or 1,
            1
        )


        # ------------------------------------------------------------------------------------------
        # CAMERA
        # ------------------------------------------------------------------------------------------

        self.cap = cv2.VideoCapture(
            CAMERA_ID
        )

        if not self.cap.isOpened():
            raise RuntimeError(
                "Could not open webcam."
            )

        self.cap.set(
            cv2.CAP_PROP_FRAME_WIDTH,
            CAMERA_WIDTH
        )

        self.cap.set(
            cv2.CAP_PROP_FRAME_HEIGHT,
            CAMERA_HEIGHT
        )

        try:
            self.cap.set(
                cv2.CAP_PROP_BUFFERSIZE,
                1
            )
        except Exception:
            pass


        # ------------------------------------------------------------------------------------------
        # TTS
        # ------------------------------------------------------------------------------------------

        self.tts = GUIAsyncTTS(
            TTS_RATE,
            TTS_VOLUME
        )


        # ------------------------------------------------------------------------------------------
        # RECOGNITION BUFFERS
        # ------------------------------------------------------------------------------------------

        self.feature_buffer = deque(
            maxlen=SEQUENCE_LENGTH
        )

        self.hand_buffer = deque(
            maxlen=SEQUENCE_LENGTH
        )

        self.model_history = deque(
            maxlen=MODEL_HISTORY_SIZE
        )

        self.last_feature = None


        # ------------------------------------------------------------------------------------------
        # PREDICTION STATE
        # ------------------------------------------------------------------------------------------

        self.model_prediction = "-"
        self.output_prediction = "NO_SIGN"

        self.confidence = 0.0
        self.margin = 0.0
        self.entropy = 1.0

        self.hand_ratio = 0.0

        self.active_token = None
        self.nosign_count = 0


        # ------------------------------------------------------------------------------------------
        # SEQUENCE
        # ------------------------------------------------------------------------------------------

        self.sentence_tokens = []

        self.token_events = []
        self.speech_events = []


        # ------------------------------------------------------------------------------------------
        # SESSION
        # ------------------------------------------------------------------------------------------

        self.running = True

        self.session_start = time.perf_counter()

        self.frame_count = 0
        self.hand_frames = 0

        self.classification_windows = 0
        self.valid_windows = 0
        self.nosign_windows = 0

        self.tokens_generated = 0

        self.sign_transition_count = 0


        # ------------------------------------------------------------------------------------------
        # FPS
        # ------------------------------------------------------------------------------------------

        self.fps_counter = 0
        self.fps_time = time.perf_counter()

        self.live_fps = 0.0


        # ------------------------------------------------------------------------------------------
        # METRICS
        # ------------------------------------------------------------------------------------------

        self.frame_metrics = []

        self.palm_times = []
        self.landmark_times = []
        self.classifier_times = []
        self.frame_processing_times = []

        self.confidence_values = []

        self.cpu_samples_raw = []
        self.cpu_samples_normalized = []

        self.ram_samples = []

        self.last_resource_sample = 0.0


        # ------------------------------------------------------------------------------------------
        # LABELLED EVALUATION
        # ------------------------------------------------------------------------------------------

        self.evaluation_samples = []

        self.expected_var = tk.StringVar(
            value="NO_SIGN"
        )


        # ------------------------------------------------------------------------------------------
        # UI VARIABLES
        # ------------------------------------------------------------------------------------------

        self.model_var = tk.StringVar(
            value="—"
        )

        self.output_var = tk.StringVar(
            value="NO_SIGN"
        )

        self.confidence_var = tk.StringVar(
            value="0.0%"
        )

        self.fps_var = tk.StringVar(
            value="0.0"
        )

        self.latency_var = tk.StringVar(
            value="0.0 ms"
        )

        self.hand_var = tk.StringVar(
            value="0%"
        )

        self.cpu_var = tk.StringVar(
            value="0%"
        )

        self.memory_var = tk.StringVar(
            value="0 MB"
        )

        self.tts_var = tk.StringVar(
            value="ON"
        )

        self.status_var = tk.StringVar(
            value="Initializing..."
        )

        self.eval_var = tk.StringVar(
            value="No labelled samples yet"
        )

        self.sequence_var = tk.StringVar(
            value="Waiting for signs..."
        )


        # ------------------------------------------------------------------------------------------
        # GUI
        # ------------------------------------------------------------------------------------------

        self.build_styles()
        self.build_gui()

        self.process.cpu_percent(
            interval=None
        )

        self.status_var.set(
            "System ready • Show a sign to the camera"
        )

        self.update_frame()


# ==================================================================================================
# 9. STYLES
# ==================================================================================================

    def build_styles(self):

        style = ttk.Style()

        try:
            style.theme_use("clam")
        except Exception:
            pass

        style.configure(
            "TCombobox",
            fieldbackground=PANEL_2,
            background=PANEL_2,
            foreground=TEXT,
            arrowcolor=TEXT,
            bordercolor=BORDER
        )


# ==================================================================================================
# 10. GUI
# ==================================================================================================

    def build_gui(self):

        # ==========================================================================================
        # HEADER
        # ==========================================================================================

        header = tk.Frame(
            self.root,
            bg=BG
        )

        header.pack(
            fill="x",
            padx=28,
            pady=(20, 12)
        )


        title_box = tk.Frame(
            header,
            bg=BG
        )

        title_box.pack(
            side="left"
        )


        tk.Label(
            title_box,
            text="CISLR",
            bg=BG,
            fg=ACCENT,
            font=("DejaVu Sans", 26, "bold")
        ).pack(
            anchor="w"
        )


        tk.Label(
            title_box,
            text="Real-Time Sign Language Interpreter",
            bg=BG,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 11)
        ).pack(
            anchor="w"
        )


        tk.Label(
            header,
            text="●  OPENVINO LIVE",
            bg=PANEL_2,
            fg=SUCCESS,
            padx=14,
            pady=8,
            font=("DejaVu Sans", 10, "bold")
        ).pack(
            side="right"
        )


        # ==========================================================================================
        # MAIN
        # ==========================================================================================

        main = tk.Frame(
            self.root,
            bg=BG
        )

        main.pack(
            fill="both",
            expand=True,
            padx=28,
            pady=5
        )

        main.grid_columnconfigure(
            0,
            weight=3
        )

        main.grid_columnconfigure(
            1,
            weight=2
        )

        main.grid_rowconfigure(
            0,
            weight=1
        )


        # ==========================================================================================
        # CAMERA
        # ==========================================================================================

        video_card = self.make_card(main)

        video_card.grid(
            row=0,
            column=0,
            sticky="nsew",
            padx=(0, 10)
        )


        video_header = tk.Frame(
            video_card,
            bg=PANEL
        )

        video_header.pack(
            fill="x",
            padx=18,
            pady=(15, 8)
        )


        tk.Label(
            video_header,
            text="LIVE CAMERA",
            bg=PANEL,
            fg=TEXT,
            font=("DejaVu Sans", 11, "bold")
        ).pack(
            side="left"
        )


        tk.Label(
            video_header,
            text="● LIVE",
            bg=PANEL,
            fg=SUCCESS,
            font=("DejaVu Sans", 9, "bold")
        ).pack(
            side="right"
        )


        self.video_label = tk.Label(
            video_card,
            bg="#020617"
        )

        self.video_label.pack(
            fill="both",
            expand=True,
            padx=14,
            pady=(0, 14)
        )


        # ==========================================================================================
        # RIGHT
        # ==========================================================================================

        right = tk.Frame(
            main,
            bg=BG
        )

        right.grid(
            row=0,
            column=1,
            sticky="nsew",
            padx=(10, 0)
        )


        # ==========================================================================================
        # CURRENT SIGN
        # ==========================================================================================

        sign_card = self.make_card(right)

        sign_card.pack(
            fill="x",
            pady=(0, 10)
        )


        tk.Label(
            sign_card,
            text="CURRENT INTERPRETATION",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 9, "bold")
        ).pack(
            anchor="w",
            padx=18,
            pady=(15, 2)
        )


        self.output_label = tk.Label(
            sign_card,
            textvariable=self.output_var,
            bg=PANEL,
            fg=WARNING,
            font=("DejaVu Sans", 27, "bold")
        )

        self.output_label.pack(
            anchor="w",
            padx=18
        )


        model_row = tk.Frame(
            sign_card,
            bg=PANEL
        )

        model_row.pack(
            fill="x",
            padx=18,
            pady=(5, 15)
        )


        tk.Label(
            model_row,
            text="Model:",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 10)
        ).pack(side="left")


        tk.Label(
            model_row,
            textvariable=self.model_var,
            bg=PANEL,
            fg=TEXT,
            font=("DejaVu Sans", 10, "bold")
        ).pack(
            side="left",
            padx=5
        )


        tk.Label(
            model_row,
            text="Confidence:",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 10)
        ).pack(
            side="left",
            padx=(20, 3)
        )


        tk.Label(
            model_row,
            textvariable=self.confidence_var,
            bg=PANEL,
            fg=ACCENT,
            font=("DejaVu Sans", 10, "bold")
        ).pack(side="left")


        # ==========================================================================================
        # TRANSLATED SEQUENCE
        # ==========================================================================================

        sequence_card = self.make_card(right)

        sequence_card.pack(
            fill="x",
            pady=(0, 10)
        )


        tk.Label(
            sequence_card,
            text="TRANSLATED SEQUENCE",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 9, "bold")
        ).pack(
            anchor="w",
            padx=18,
            pady=(15, 5)
        )


        tk.Label(
            sequence_card,
            textvariable=self.sequence_var,
            bg=PANEL,
            fg=TEXT,
            wraplength=460,
            justify="left",
            font=("DejaVu Sans", 15, "bold")
        ).pack(
            anchor="w",
            padx=18,
            pady=(0, 15)
        )


        # ==========================================================================================
        # PERFORMANCE
        # ==========================================================================================

        metrics_card = self.make_card(right)

        metrics_card.pack(
            fill="x",
            pady=(0, 10)
        )


        tk.Label(
            metrics_card,
            text="LIVE PERFORMANCE",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 9, "bold")
        ).pack(
            anchor="w",
            padx=18,
            pady=(15, 10)
        )


        metric_grid = tk.Frame(
            metrics_card,
            bg=PANEL
        )

        metric_grid.pack(
            fill="x",
            padx=15,
            pady=(0, 15)
        )

        metric_grid.grid_columnconfigure(
            (0, 1, 2),
            weight=1
        )


        self.metric_box(
            metric_grid,
            0,
            0,
            "FPS",
            self.fps_var
        )

        self.metric_box(
            metric_grid,
            0,
            1,
            "FRAME LATENCY",
            self.latency_var
        )

        self.metric_box(
            metric_grid,
            0,
            2,
            "HAND",
            self.hand_var
        )

        self.metric_box(
            metric_grid,
            1,
            0,
            "CPU",
            self.cpu_var
        )

        self.metric_box(
            metric_grid,
            1,
            1,
            "MEMORY",
            self.memory_var
        )

        self.metric_box(
            metric_grid,
            1,
            2,
            "TTS",
            self.tts_var
        )


        # ==========================================================================================
        # L32 EVALUATION
        # ==========================================================================================

        eval_card = self.make_card(right)

        eval_card.pack(
            fill="x",
            pady=(0, 10)
        )


        tk.Label(
            eval_card,
            text="L32 • LABELLED EVALUATION",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 9, "bold")
        ).pack(
            anchor="w",
            padx=18,
            pady=(15, 7)
        )


        eval_row = tk.Frame(
            eval_card,
            bg=PANEL
        )

        eval_row.pack(
            fill="x",
            padx=18
        )


        expected_values = (
            ["NO_SIGN"]
            +
            [
                x.upper()
                for x in CLASS_NAMES
            ]
        )


        self.expected_combo = ttk.Combobox(
            eval_row,
            textvariable=self.expected_var,
            values=expected_values,
            state="readonly",
            width=17
        )

        self.expected_combo.pack(
            side="left"
        )


        self.make_button(
            eval_row,
            "Evaluate Current",
            self.evaluate_current,
            ACCENT
        ).pack(
            side="left",
            padx=8
        )


        tk.Label(
            eval_card,
            textvariable=self.eval_var,
            bg=PANEL,
            fg=TEXT,
            font=("DejaVu Sans", 9)
        ).pack(
            anchor="w",
            padx=18,
            pady=(8, 15)
        )


        # ==========================================================================================
        # CONTROLS
        # ==========================================================================================

        controls = self.make_card(right)

        controls.pack(fill="x")


        tk.Label(
            controls,
            text="CONTROLS",
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 9, "bold")
        ).pack(
            anchor="w",
            padx=18,
            pady=(15, 8)
        )


        buttons1 = tk.Frame(
            controls,
            bg=PANEL
        )

        buttons1.pack(
            fill="x",
            padx=15,
            pady=(0, 7)
        )


        self.make_button(
            buttons1,
            "Speak",
            self.speak_sequence,
            ACCENT
        ).pack(
            side="left",
            expand=True,
            fill="x",
            padx=3
        )


        self.mute_button = self.make_button(
            buttons1,
            "Mute",
            self.toggle_mute,
            PURPLE
        )

        self.mute_button.pack(
            side="left",
            expand=True,
            fill="x",
            padx=3
        )


        buttons2 = tk.Frame(
            controls,
            bg=PANEL
        )

        buttons2.pack(
            fill="x",
            padx=15,
            pady=(0, 7)
        )


        self.make_button(
            buttons2,
            "Clear",
            self.clear_sequence,
            WARNING
        ).pack(
            side="left",
            expand=True,
            fill="x",
            padx=3
        )


        self.make_button(
            buttons2,
            "Undo",
            self.delete_last,
            PANEL_2
        ).pack(
            side="left",
            expand=True,
            fill="x",
            padx=3
        )


        buttons3 = tk.Frame(
            controls,
            bg=PANEL
        )

        buttons3.pack(
            fill="x",
            padx=15,
            pady=(0, 15)
        )


        self.make_button(
            buttons3,
            "Reset Recognition",
            self.reset_recognition,
            PANEL_2
        ).pack(
            side="left",
            expand=True,
            fill="x",
            padx=3
        )


        self.make_button(
            buttons3,
            "Finish & Save Report",
            self.finish_session,
            SUCCESS
        ).pack(
            side="left",
            expand=True,
            fill="x",
            padx=3
        )


        # ==========================================================================================
        # STATUS
        # ==========================================================================================

        status = tk.Frame(
            self.root,
            bg=PANEL
        )

        status.pack(
            fill="x",
            side="bottom"
        )


        tk.Label(
            status,
            textvariable=self.status_var,
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 9),
            padx=20,
            pady=8
        ).pack(
            side="left"
        )


        tk.Label(
            status,
            text=(
                "L26C Model • L27 NO_SIGN • "
                "Model-Direct Sequence • TTS • GUI • Benchmark"
            ),
            bg=PANEL,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 8),
            padx=20
        ).pack(
            side="right"
        )


# ==================================================================================================
# 11. CARD
# ==================================================================================================

    def make_card(self, parent):

        return tk.Frame(
            parent,
            bg=PANEL,
            highlightbackground=BORDER,
            highlightthickness=1
        )


# ==================================================================================================
# 12. BUTTON
# ==================================================================================================

    def make_button(
        self,
        parent,
        text,
        command,
        color
    ):

        return tk.Button(
            parent,
            text=text,
            command=command,
            bg=color,
            fg=TEXT,
            activebackground=color,
            activeforeground=TEXT,
            relief="flat",
            bd=0,
            padx=10,
            pady=8,
            cursor="hand2",
            font=("DejaVu Sans", 9, "bold")
        )


# ==================================================================================================
# 13. METRIC BOX
# ==================================================================================================

    def metric_box(
        self,
        parent,
        row,
        column,
        title,
        variable
    ):

        box = tk.Frame(
            parent,
            bg=PANEL_2
        )

        box.grid(
            row=row,
            column=column,
            sticky="nsew",
            padx=3,
            pady=3
        )


        tk.Label(
            box,
            text=title,
            bg=PANEL_2,
            fg=TEXT_SECONDARY,
            font=("DejaVu Sans", 7, "bold")
        ).pack(
            pady=(7, 1)
        )


        tk.Label(
            box,
            textvariable=variable,
            bg=PANEL_2,
            fg=TEXT,
            font=("DejaVu Sans", 11, "bold")
        ).pack(
            pady=(0, 7)
        )


# ==================================================================================================
# 14. MODEL CONFIRMATION
# ==================================================================================================

    def confirmed_model_prediction(self):

        if len(self.model_history) < MODEL_CONFIRM_VOTES:
            return None

        counts = Counter(
            self.model_history
        )

        candidate, votes = (
            counts.most_common(1)[0]
        )

        if votes >= MODEL_CONFIRM_VOTES:
            return candidate

        return None


# ==================================================================================================
# 15. CREATE TOKEN
# ==================================================================================================

    def create_token(
        self,
        sign,
        transition_from=None
    ):

        self.active_token = sign

        self.sentence_tokens.append(
            sign
        )


        if len(self.sentence_tokens) > MAX_SENTENCE_TOKENS:

            self.sentence_tokens = (
                self.sentence_tokens[
                    -MAX_SENTENCE_TOKENS:
                ]
            )


        self.tokens_generated += 1


        if transition_from is not None:

            self.sign_transition_count += 1


        event = {

            "event_index":
                self.tokens_generated,

            "time_seconds":
                time.perf_counter()
                -
                self.session_start,

            "frame":
                self.frame_count,

            "token":
                sign,

            "transition_from":
                transition_from,

            "model_prediction":
                self.model_prediction,

            "confidence":
                self.confidence,

            "margin":
                self.margin,

            "entropy":
                self.entropy,

            "hand_ratio":
                self.hand_ratio
        }


        self.token_events.append(
            event
        )


        self.sequence_var.set(
            " ".join(
                self.sentence_tokens
            )
        )


        # ------------------------------------------------------------------------------------------
        # TTS NEW TOKEN
        # ------------------------------------------------------------------------------------------

        self.tts.speak(
            sign
        )


        self.speech_events.append({

            "time_seconds":
                time.perf_counter()
                -
                self.session_start,

            "text":
                sign,

            "source":
                "automatic_token"
        })


        if transition_from is None:

            self.status_var.set(
                f'Recognized "{sign.upper()}"'
            )

        else:

            self.status_var.set(
                f'{transition_from.upper()} → {sign.upper()}'
            )


# ==================================================================================================
# 16. FRAME UPDATE
# ==================================================================================================

    def update_frame(self):

        if not self.running:
            return


        frame_start = time.perf_counter()


        ok, frame = self.cap.read()


        if not ok:

            self.root.after(
                GUI_INTERVAL_MS,
                self.update_frame
            )

            return


        self.frame_count += 1


        # ==========================================================================================
        # HAND DETECTION
        # ==========================================================================================

        try:

            results, palm_ms, landmark_ms = detect_hands(
                frame
            )

        except Exception as e:

            print(
                "[DETECTION ERROR]",
                e
            )

            results = []

            palm_ms = 0.0
            landmark_ms = 0.0


        current_feature = None
        hand_present = False


        if results:

            current_feature = normalize_for_classifier(
                results[0]["landmarks"]
            )


            if current_feature is not None:

                hand_present = True

                self.hand_frames += 1

                self.last_feature = (
                    current_feature.copy()
                )


        # ==========================================================================================
        # HAND BUFFER
        # ==========================================================================================

        self.hand_buffer.append(
            int(hand_present)
        )


        self.hand_ratio = (
            float(
                np.mean(
                    self.hand_buffer
                )
            )
            if self.hand_buffer
            else 0.0
        )


        # ==========================================================================================
        # FEATURE BUFFER
        # ==========================================================================================

        if current_feature is not None:

            self.feature_buffer.append(
                current_feature.copy()
            )

        elif self.last_feature is not None:

            self.feature_buffer.append(
                self.last_feature.copy()
            )


        classifier_ms = 0.0


        # Default each frame.

        self.output_prediction = "NO_SIGN"


        # ==========================================================================================
        # CLASSIFIER
        # ==========================================================================================

        if len(self.feature_buffer) == SEQUENCE_LENGTH:

            metrics = classify_sequence(
                np.stack(
                    self.feature_buffer,
                    axis=0
                )
            )


            self.classification_windows += 1


            # --------------------------------------------------------------------------------------
            # MODEL IS AUTHORITATIVE
            # --------------------------------------------------------------------------------------

            self.model_prediction = (
                metrics["prediction"]
            )

            self.confidence = float(
                metrics["confidence"]
            )

            self.margin = float(
                metrics["margin"]
            )

            self.entropy = float(
                metrics["entropy"]
            )

            classifier_ms = float(
                metrics["inference_ms"]
            )


            self.confidence_values.append(
                self.confidence
            )


            # ======================================================================================
            # L27
            #
            # ONLY:
            #
            # model class -> accepted
            #
            # OR
            #
            # model class -> NO_SIGN
            #
            # ======================================================================================

            rejection = l27_reject(
                metrics,
                self.hand_ratio
            )


            is_nosign = (
                rejection
                ==
                "NO_SIGN"
            )


            # ======================================================================================
            # NO_SIGN
            # ======================================================================================

            if is_nosign:

                self.output_prediction = (
                    "NO_SIGN"
                )


                self.nosign_windows += 1

                self.nosign_count += 1


                # Remove stale votes.

                self.model_history.clear()


                # Release held sign.

                if (
                    self.nosign_count
                    >=
                    NO_SIGN_RELEASE_FRAMES
                ):

                    self.active_token = None


            # ======================================================================================
            # VALID SIGN
            # ======================================================================================

            else:

                self.valid_windows += 1

                self.nosign_count = 0


                # ==================================================================================
                # CURRENT INTERPRETATION IS DIRECTLY THE MODEL
                # ==================================================================================

                self.output_prediction = (
                    self.model_prediction
                )


                # ==================================================================================
                # MODEL VOTE HISTORY
                # ==================================================================================

                self.model_history.append(
                    self.model_prediction
                )


                confirmed = (
                    self.confirmed_model_prediction()
                )


                # ==================================================================================
                # UPDATED TOKEN LOGIC
                # ==================================================================================
                #
                # Case A
                #
                # active = None
                # confirmed = MOTHER
                #
                # -> add MOTHER
                #
                #
                # Case B
                #
                # active = MOTHER
                # confirmed = MOTHER
                #
                # -> do nothing
                #
                #
                # Case C
                #
                # active = FRIDAY
                # confirmed = MOTHER
                #
                # -> immediately add MOTHER
                #
                # NO_SIGN IS NOT REQUIRED.
                #
                # ==================================================================================

                if confirmed is not None:


                    # ------------------------------------------------------------------------------
                    # FIRST TOKEN
                    # ------------------------------------------------------------------------------

                    if self.active_token is None:

                        self.create_token(
                            confirmed
                        )


                        # Clear history after event.
                        # New observations now build the next decision.

                        self.model_history.clear()


                    # ------------------------------------------------------------------------------
                    # SAME SIGN STILL HELD
                    # ------------------------------------------------------------------------------

                    elif (
                        confirmed
                        ==
                        self.active_token
                    ):

                        # No duplicate token.

                        pass


                    # ------------------------------------------------------------------------------
                    # DIFFERENT CONFIRMED SIGN
                    # ------------------------------------------------------------------------------

                    else:

                        previous = (
                            self.active_token
                        )


                        self.create_token(
                            confirmed,
                            transition_from=previous
                        )


                        print(
                            f"[SIGN CHANGE] "
                            f"{previous.upper()} "
                            f"-> "
                            f"{confirmed.upper()}"
                        )


                        # Very important:
                        #
                        # Remove old votes so the previous class
                        # cannot dominate the next transition.

                        self.model_history.clear()


        # ==========================================================================================
        # DRAW HAND
        # ==========================================================================================

        display = frame.copy()


        for result in results:

            try:
                draw_hand(
                    display,
                    result
                )
            except Exception:
                pass


        # ==========================================================================================
        # PROCESSING LATENCY
        # ==========================================================================================

        processing_ms = (
            time.perf_counter()
            -
            frame_start
        ) * 1000.0


        # ==========================================================================================
        # FPS
        # ==========================================================================================

        self.fps_counter += 1

        now = time.perf_counter()

        elapsed = (
            now -
            self.fps_time
        )


        if elapsed >= 0.5:

            self.live_fps = (
                self.fps_counter
                /
                elapsed
            )

            self.fps_counter = 0
            self.fps_time = now


        # ==========================================================================================
        # CPU / RAM
        # ==========================================================================================

        cpu_raw = (
            self.cpu_samples_raw[-1]
            if self.cpu_samples_raw
            else 0.0
        )

        cpu_normalized = (
            self.cpu_samples_normalized[-1]
            if self.cpu_samples_normalized
            else 0.0
        )

        memory_mb = (
            self.ram_samples[-1]
            if self.ram_samples
            else 0.0
        )


        if (
            now -
            self.last_resource_sample
            >=
            RESOURCE_SAMPLE_INTERVAL
        ):

            cpu_raw = float(
                self.process.cpu_percent(
                    interval=None
                )
            )


            # User-friendly CPU %
            #
            # Raw process CPU can exceed 100% because
            # multiple logical CPU cores can be used.

            cpu_normalized = min(
                100.0,
                cpu_raw
                /
                self.logical_cpu_count
            )


            memory_mb = float(
                self.process.memory_info().rss
                /
                (1024 ** 2)
            )


            self.cpu_samples_raw.append(
                cpu_raw
            )

            self.cpu_samples_normalized.append(
                cpu_normalized
            )

            self.ram_samples.append(
                memory_mb
            )

            self.last_resource_sample = now


        # ==========================================================================================
        # METRICS
        # ==========================================================================================

        self.palm_times.append(
            palm_ms
        )

        self.landmark_times.append(
            landmark_ms
        )

        if classifier_ms > 0:

            self.classifier_times.append(
                classifier_ms
            )

        self.frame_processing_times.append(
            processing_ms
        )


        metric_row = {

            "frame":
                self.frame_count,

            "time_seconds":
                now
                -
                self.session_start,

            "hand_present":
                int(hand_present),

            "hand_ratio":
                self.hand_ratio,

            "model_prediction":
                self.model_prediction,

            "output_prediction":
                self.output_prediction,

            "active_token":
                (
                    self.active_token
                    if self.active_token
                    else ""
                ),

            "confidence":
                self.confidence,

            "margin":
                self.margin,

            "entropy":
                self.entropy,

            "palm_ms":
                palm_ms,

            "landmark_ms":
                landmark_ms,

            "classifier_ms":
                classifier_ms,

            "frame_processing_ms":
                processing_ms,

            "live_fps":
                self.live_fps,

            "cpu_raw_percent":
                cpu_raw,

            "cpu_normalized_percent":
                cpu_normalized,

            "memory_mb":
                memory_mb
        }


        if len(self.frame_metrics) < MAX_METRIC_HISTORY:

            self.frame_metrics.append(
                metric_row
            )


        # ==========================================================================================
        # GUI VALUES
        # ==========================================================================================

        self.model_var.set(
            self.model_prediction.upper()
        )


        self.output_var.set(
            self.output_prediction.upper()
        )


        self.confidence_var.set(
            f"{self.confidence * 100:.1f}%"
        )


        self.fps_var.set(
            f"{self.live_fps:.1f}"
        )


        self.latency_var.set(
            f"{processing_ms:.1f} ms"
        )


        self.hand_var.set(
            f"{self.hand_ratio * 100:.0f}%"
        )


        self.cpu_var.set(
            f"{cpu_normalized:.1f}%"
        )


        self.memory_var.set(
            f"{memory_mb:.0f} MB"
        )


        self.tts_var.set(
            "MUTED"
            if self.tts.muted
            else "ON"
        )


        # ==========================================================================================
        # OUTPUT COLOR
        # ==========================================================================================

        if self.output_prediction == "NO_SIGN":

            self.output_label.configure(
                fg=WARNING
            )

        else:

            self.output_label.configure(
                fg=SUCCESS
            )


        # ==========================================================================================
        # CAMERA IMAGE
        # ==========================================================================================

        display = cv2.resize(
            display,
            (760, 570)
        )


        rgb = cv2.cvtColor(
            display,
            cv2.COLOR_BGR2RGB
        )


        pil_image = Image.fromarray(
            rgb
        )


        photo = ImageTk.PhotoImage(
            image=pil_image
        )


        self.video_label.configure(
            image=photo
        )


        self.video_label.image = photo


        # ==========================================================================================
        # NEXT FRAME
        # ==========================================================================================

        self.root.after(
            GUI_INTERVAL_MS,
            self.update_frame
        )


# ==================================================================================================
# 17. SPEAK
# ==================================================================================================

    def speak_sequence(self):

        sentence = " ".join(
            self.sentence_tokens
        )


        if not sentence:

            self.status_var.set(
                "Nothing to speak yet"
            )

            return


        self.tts.speak(
            sentence
        )


        self.speech_events.append({

            "time_seconds":
                time.perf_counter()
                -
                self.session_start,

            "text":
                sentence,

            "source":
                "manual_full_sequence"
        })


        self.status_var.set(
            "Speaking translated sequence"
        )


# ==================================================================================================
# 18. MUTE
# ==================================================================================================

    def toggle_mute(self):

        muted = self.tts.toggle_mute()


        self.tts_var.set(
            "MUTED"
            if muted
            else "ON"
        )


        self.mute_button.configure(
            text=(
                "Unmute"
                if muted
                else "Mute"
            )
        )


        self.status_var.set(
            "Speech muted"
            if muted
            else "Speech enabled"
        )


# ==================================================================================================
# 19. CLEAR
# ==================================================================================================

    def clear_sequence(self):

        self.sentence_tokens.clear()

        self.sequence_var.set(
            "Waiting for signs..."
        )

        self.reset_recognition()

        self.tts.clear()

        self.status_var.set(
            "Translated sequence cleared"
        )


# ==================================================================================================
# 20. UNDO
# ==================================================================================================

    def delete_last(self):

        if not self.sentence_tokens:

            self.status_var.set(
                "No token to remove"
            )

            return


        removed = self.sentence_tokens.pop()


        self.sequence_var.set(
            " ".join(
                self.sentence_tokens
            )
            if self.sentence_tokens
            else "Waiting for signs..."
        )


        # Keep active state consistent.

        self.active_token = (
            self.sentence_tokens[-1]
            if self.sentence_tokens
            else None
        )


        self.model_history.clear()


        self.status_var.set(
            f'Removed "{removed.upper()}"'
        )


# ==================================================================================================
# 21. RESET RECOGNITION
# ==================================================================================================

    def reset_recognition(self):

        self.feature_buffer.clear()
        self.hand_buffer.clear()
        self.model_history.clear()

        self.last_feature = None

        self.active_token = None

        self.nosign_count = 0

        self.model_prediction = "-"

        self.output_prediction = "NO_SIGN"

        self.confidence = 0.0
        self.margin = 0.0
        self.entropy = 1.0

        self.hand_ratio = 0.0

        self.status_var.set(
            "Recognition state reset"
        )


# ==================================================================================================
# 22. LABELLED EVALUATION
# ==================================================================================================

    def evaluate_current(self):

        expected = (
            self.expected_var
            .get()
            .strip()
            .upper()
        )


        predicted = (
            self.output_prediction
            .strip()
            .upper()
        )


        correct = (
            expected
            ==
            predicted
        )


        sample = {

            "sample_id":
                len(
                    self.evaluation_samples
                )
                +
                1,

            "time_seconds":
                time.perf_counter()
                -
                self.session_start,

            "expected":
                expected,

            "predicted":
                predicted,

            "model_prediction":
                self.model_prediction.upper(),

            "confidence":
                self.confidence,

            "margin":
                self.margin,

            "entropy":
                self.entropy,

            "hand_ratio":
                self.hand_ratio,

            "correct":
                bool(correct)
        }


        self.evaluation_samples.append(
            sample
        )


        correct_count = sum(
            int(x["correct"])
            for x in self.evaluation_samples
        )


        total = len(
            self.evaluation_samples
        )


        accuracy = (
            100.0
            *
            correct_count
            /
            total
        )


        self.eval_var.set(
            f"{correct_count}/{total} correct • "
            f"{accuracy:.1f}% live accuracy"
        )


        self.status_var.set(
            (
                f"Sample #{total}: "
                f"{'CORRECT' if correct else 'INCORRECT'}"
            )
        )


# ==================================================================================================
# 23. SAVE FRAME CSV
# ==================================================================================================

    def save_frame_csv(self):

        path = (
            L32_REPORT_ROOT /
            "l32_frame_metrics.csv"
        )


        if not self.frame_metrics:

            return path


        fields = list(
            self.frame_metrics[0].keys()
        )


        with open(
            path,
            "w",
            newline=""
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=fields
            )

            writer.writeheader()

            writer.writerows(
                self.frame_metrics
            )


        return path


# ==================================================================================================
# 24. SUMMARY
# ==================================================================================================

    def build_summary(self):

        duration = (
            time.perf_counter()
            -
            self.session_start
        )


        overall_fps = (
            self.frame_count
            /
            duration
            if duration > 0
            else 0.0
        )


        hand_rate = (
            self.hand_frames
            /
            self.frame_count
            if self.frame_count > 0
            else 0.0
        )


        # ==========================================================================================
        # LABELLED EVALUATION
        # ==========================================================================================

        labelled_n = len(
            self.evaluation_samples
        )


        labelled_correct = sum(
            int(x["correct"])
            for x in self.evaluation_samples
        )


        labelled_accuracy = (
            labelled_correct
            /
            labelled_n
            if labelled_n > 0
            else None
        )


        # ==========================================================================================
        # NO-HAND ROBUSTNESS
        # ==========================================================================================

        no_hand_rows = [
            row
            for row in self.frame_metrics
            if row["hand_present"] == 0
        ]


        no_hand_rejected = [
            row
            for row in no_hand_rows
            if row["output_prediction"]
            ==
            "NO_SIGN"
        ]


        no_hand_rejection_rate = (
            len(no_hand_rejected)
            /
            len(no_hand_rows)
            if no_hand_rows
            else None
        )


        # ==========================================================================================
        # SUMMARY
        # ==========================================================================================

        return {

            "stage":
                "L31_L32",

            "status":
                "DEVELOPMENT_EVALUATION_COMPLETE",

            "timestamp":
                datetime.now().isoformat(),


            "session": {

                "duration_seconds":
                    duration,

                "frames":
                    self.frame_count,

                "overall_fps":
                    overall_fps,

                "hand_frames":
                    self.hand_frames,

                "hand_detection_rate":
                    hand_rate,

                "classification_windows":
                    self.classification_windows,

                "valid_sign_windows":
                    self.valid_windows,

                "nosign_windows":
                    self.nosign_windows,

                "tokens_generated":
                    self.tokens_generated,

                "sign_transitions":
                    self.sign_transition_count,

                "final_tokens":
                    list(
                        self.sentence_tokens
                    ),

                "final_sequence":
                    " ".join(
                        self.sentence_tokens
                    )
            },


            "latency_ms": {

                "palm_mean":
                    safe_mean(
                        self.palm_times
                    ),

                "palm_median":
                    safe_median(
                        self.palm_times
                    ),

                "palm_p95":
                    safe_percentile(
                        self.palm_times,
                        95
                    ),

                "landmark_mean":
                    safe_mean(
                        self.landmark_times
                    ),

                "landmark_median":
                    safe_median(
                        self.landmark_times
                    ),

                "landmark_p95":
                    safe_percentile(
                        self.landmark_times,
                        95
                    ),

                "classifier_mean":
                    safe_mean(
                        self.classifier_times
                    ),

                "classifier_median":
                    safe_median(
                        self.classifier_times
                    ),

                "classifier_p95":
                    safe_percentile(
                        self.classifier_times,
                        95
                    ),

                "frame_processing_mean":
                    safe_mean(
                        self.frame_processing_times
                    ),

                "frame_processing_median":
                    safe_median(
                        self.frame_processing_times
                    ),

                "frame_processing_p95":
                    safe_percentile(
                        self.frame_processing_times,
                        95
                    )
            },


            "resources": {

                "logical_cpu_count":
                    self.logical_cpu_count,

                "cpu_raw_percent_mean":
                    safe_mean(
                        self.cpu_samples_raw
                    ),

                "cpu_normalized_percent_mean":
                    safe_mean(
                        self.cpu_samples_normalized
                    ),

                "cpu_normalized_percent_p95":
                    safe_percentile(
                        self.cpu_samples_normalized,
                        95
                    ),

                "memory_mb_mean":
                    safe_mean(
                        self.ram_samples
                    ),

                "memory_mb_peak":
                    (
                        max(self.ram_samples)
                        if self.ram_samples
                        else 0.0
                    )
            },


            "confidence": {

                "mean":
                    safe_mean(
                        self.confidence_values
                    ),

                "median":
                    safe_median(
                        self.confidence_values
                    ),

                "minimum":
                    (
                        min(self.confidence_values)
                        if self.confidence_values
                        else 0.0
                    ),

                "maximum":
                    (
                        max(self.confidence_values)
                        if self.confidence_values
                        else 0.0
                    )
            },


            "labelled_evaluation": {

                "samples":
                    labelled_n,

                "correct":
                    labelled_correct,

                "accuracy":
                    labelled_accuracy,

                "accuracy_percent":
                    (
                        labelled_accuracy * 100.0
                        if labelled_accuracy is not None
                        else None
                    ),

                "note":
                    (
                        "Manually labelled live deployment "
                        "evaluation; not locked L18B test."
                    )
            },


            "robustness": {

                "no_hand_frames":
                    len(
                        no_hand_rows
                    ),

                "no_hand_frames_rejected":
                    len(
                        no_hand_rejected
                    ),

                "no_hand_rejection_rate":
                    no_hand_rejection_rate,

                "no_hand_rejection_percent":
                    (
                        no_hand_rejection_rate * 100.0
                        if no_hand_rejection_rate is not None
                        else None
                    )
            },


            "integrity": {

                "sign_identity_source":
                    "L26C trained OpenVINO BiGRU",

                "l27_role":
                    "NO_SIGN rejection only",

                "different_valid_sign_requires_nosign":
                    False,

                "same_sign_duplicate_suppression":
                    True,

                "locked_test_used":
                    False,

                "training":
                    False,

                "retraining":
                    False
            }
        }


# ==================================================================================================
# 25. SAVE SESSION
# ==================================================================================================

    def save_session(self):

        frame_csv = (
            self.save_frame_csv()
        )


        # ------------------------------------------------------------------------------------------
        # TOKENS
        # ------------------------------------------------------------------------------------------

        token_path = (
            L31_REPORT_ROOT /
            "l31_token_events.json"
        )


        with open(
            token_path,
            "w"
        ) as f:

            json.dump(
                self.token_events,
                f,
                indent=2
            )


        # ------------------------------------------------------------------------------------------
        # SPEECH
        # ------------------------------------------------------------------------------------------

        speech_path = (
            L31_REPORT_ROOT /
            "l31_speech_events.json"
        )


        with open(
            speech_path,
            "w"
        ) as f:

            json.dump(
                self.speech_events,
                f,
                indent=2
            )


        # ------------------------------------------------------------------------------------------
        # LABELLED EVALUATION
        # ------------------------------------------------------------------------------------------

        evaluation_path = (
            L32_REPORT_ROOT /
            "l32_labelled_evaluation.json"
        )


        with open(
            evaluation_path,
            "w"
        ) as f:

            json.dump(
                self.evaluation_samples,
                f,
                indent=2
            )


        # ------------------------------------------------------------------------------------------
        # SUMMARY
        # ------------------------------------------------------------------------------------------

        summary = (
            self.build_summary()
        )


        summary_path = (
            L32_REPORT_ROOT /
            "l32_summary.json"
        )


        with open(
            summary_path,
            "w"
        ) as f:

            json.dump(
                summary,
                f,
                indent=2
            )


        # ------------------------------------------------------------------------------------------
        # CONFIG
        # ------------------------------------------------------------------------------------------

        config = {

            "camera_id":
                CAMERA_ID,

            "camera_width":
                CAMERA_WIDTH,

            "camera_height":
                CAMERA_HEIGHT,

            "sequence_length":
                SEQUENCE_LENGTH,

            "feature_dim":
                FEATURE_DIM,

            "model_history_size":
                MODEL_HISTORY_SIZE,

            "model_confirm_votes":
                MODEL_CONFIRM_VOTES,

            "nosign_release_frames":
                NO_SIGN_RELEASE_FRAMES,

            "different_sign_requires_nosign":
                False,

            "same_sign_duplicate_suppression":
                True,

            "vocabulary":
                CLASS_NAMES,

            "tts_rate":
                TTS_RATE,

            "tts_volume":
                TTS_VOLUME
        }


        config_path = (
            L32_CONFIG_ROOT /
            "l32_config.json"
        )


        with open(
            config_path,
            "w"
        ) as f:

            json.dump(
                config,
                f,
                indent=2
            )


        # ------------------------------------------------------------------------------------------
        # MASTER
        # ------------------------------------------------------------------------------------------

        master = {

            "stage":
                "L31_L32",

            "status":
                "DEVELOPMENT_EVALUATION_COMPLETE",

            "pipeline": (
                "webcam -> OpenVINO palm -> "
                "OpenVINO landmarks -> "
                "L26C model -> "
                "L27 NO_SIGN gate -> "
                "continuous model-direct token sequence -> "
                "TTS -> GUI -> evaluation"
            ),

            "sequence_logic": {

                "sign_identity":
                    "L26C model",

                "nosign_gate":
                    "L27",

                "confirmation":
                    "3-of-5 model predictions",

                "different_valid_sign_transition":
                    "allowed without NO_SIGN",

                "same_sign_held":
                    "duplicate suppressed"
            },

            "artifacts": {

                "frame_metrics_csv":
                    str(frame_csv),

                "token_events":
                    str(token_path),

                "speech_events":
                    str(speech_path),

                "labelled_evaluation":
                    str(evaluation_path),

                "summary":
                    str(summary_path),

                "config":
                    str(config_path)
            },

            "integrity": {

                "locked_test_used":
                    False,

                "training":
                    False,

                "retraining":
                    False,

                "model_direct_sign_identity":
                    True,

                "l27_nosign_only":
                    True
            }
        }


        master_path = (
            L32_ROOT /
            "L31_L32_MASTER.json"
        )


        with open(
            master_path,
            "w"
        ) as f:

            json.dump(
                master,
                f,
                indent=2
            )


        with open(
            master_path,
            "rb"
        ) as f:

            master_sha = hashlib.sha256(
                f.read()
            ).hexdigest()


        return (
            summary,
            master_path,
            master_sha
        )


# ==================================================================================================
# 26. FINISH
# ==================================================================================================

    def finish_session(self):

        if not self.running:
            return


        self.running = False


        summary, master_path, master_sha = (
            self.save_session()
        )


        self.cap.release()

        self.tts.stop()


        session = summary["session"]

        latency = summary["latency_ms"]

        resources = summary["resources"]

        labelled = summary[
            "labelled_evaluation"
        ]

        robustness = summary[
            "robustness"
        ]


        print()
        print("=" * 110)
        print("L31 + L32 — FINAL DEVELOPMENT EVALUATION")
        print("=" * 110)


        print()
        print("SESSION")
        print(
            "Frames                  :",
            session["frames"]
        )

        print(
            "Duration                :",
            f'{session["duration_seconds"]:.2f} sec'
        )

        print(
            "Overall FPS             :",
            f'{session["overall_fps"]:.2f}'
        )

        print(
            "Hand detection rate     :",
            f'{session["hand_detection_rate"] * 100:.2f}%'
        )


        print()
        print("SEQUENCE")

        print(
            "Tokens generated        :",
            session["tokens_generated"]
        )

        print(
            "Direct sign transitions :",
            session["sign_transitions"]
        )

        print(
            "Final sequence          :",
            session["final_sequence"]
        )


        print()
        print("PALM LATENCY")

        print(
            "Mean                    :",
            f'{latency["palm_mean"]:.3f} ms'
        )

        print(
            "Median                  :",
            f'{latency["palm_median"]:.3f} ms'
        )

        print(
            "P95                     :",
            f'{latency["palm_p95"]:.3f} ms'
        )


        print()
        print("LANDMARK LATENCY")

        print(
            "Mean                    :",
            f'{latency["landmark_mean"]:.3f} ms'
        )

        print(
            "Median                  :",
            f'{latency["landmark_median"]:.3f} ms'
        )

        print(
            "P95                     :",
            f'{latency["landmark_p95"]:.3f} ms'
        )


        print()
        print("L26C CLASSIFIER LATENCY")

        print(
            "Mean                    :",
            f'{latency["classifier_mean"]:.3f} ms'
        )

        print(
            "Median                  :",
            f'{latency["classifier_median"]:.3f} ms'
        )

        print(
            "P95                     :",
            f'{latency["classifier_p95"]:.3f} ms'
        )


        print()
        print("FRAME PROCESSING")

        print(
            "Mean                    :",
            f'{latency["frame_processing_mean"]:.3f} ms'
        )

        print(
            "Median                  :",
            f'{latency["frame_processing_median"]:.3f} ms'
        )

        print(
            "P95                     :",
            f'{latency["frame_processing_p95"]:.3f} ms'
        )


        print()
        print("RESOURCES")

        print(
            "Mean CPU                :",
            f'{resources["cpu_normalized_percent_mean"]:.2f}%'
        )

        print(
            "P95 CPU                 :",
            f'{resources["cpu_normalized_percent_p95"]:.2f}%'
        )

        print(
            "Mean RAM                :",
            f'{resources["memory_mb_mean"]:.2f} MB'
        )

        print(
            "Peak RAM                :",
            f'{resources["memory_mb_peak"]:.2f} MB'
        )


        print()
        print("NO_SIGN ROBUSTNESS")

        print(
            "No-hand frames          :",
            robustness["no_hand_frames"]
        )

        if (
            robustness[
                "no_hand_rejection_percent"
            ]
            is not None
        ):

            print(
                "No-hand rejection       :",
                f'{robustness["no_hand_rejection_percent"]:.2f}%'
            )

        else:

            print(
                "No-hand rejection       :",
                "NOT MEASURED"
            )


        print()
        print("LABELLED LIVE EVALUATION")

        if labelled["samples"] > 0:

            print(
                "Samples                 :",
                labelled["samples"]
            )

            print(
                "Correct                 :",
                labelled["correct"]
            )

            print(
                "Accuracy                :",
                f'{labelled["accuracy_percent"]:.2f}%'
            )

        else:

            print(
                "Samples                 : 0"
            )

            print(
                "Accuracy                : NOT MEASURED"
            )


        print()
        print("INTEGRITY")
        print("Locked L18B test used   : NO")
        print("Training                : NO")
        print("Retraining              : NO")
        print("Sign identity           : L26C MODEL")
        print("L27 role                : NO_SIGN ONLY")


        print()
        print("Master:")
        print(master_path)

        print()
        print("Master SHA256:")
        print(master_sha)

        print()
        print("=" * 110)


        try:

            messagebox.showinfo(
                "CISLR Evaluation Saved",
                (
                    "L31 + L32 report saved.\n\n"
                    f"FPS: {session['overall_fps']:.2f}\n"
                    f"Frame latency: "
                    f"{latency['frame_processing_mean']:.2f} ms\n"
                    f"Hand detection: "
                    f"{session['hand_detection_rate']*100:.1f}%\n"
                    f"Tokens: "
                    f"{session['tokens_generated']}\n\n"
                    "No locked test data was used."
                )
            )

        except Exception:

            pass


        self.root.destroy()


# ==================================================================================================
# 27. CLOSE
# ==================================================================================================

    def close(self):

        if not self.running:

            try:
                self.root.destroy()
            except Exception:
                pass

            return


        answer = messagebox.askyesno(
            "Finish CISLR Session",
            (
                "Finish the session and save "
                "the L31/L32 report?"
            )
        )


        if answer:
            self.finish_session()


# ==================================================================================================
# 28. START
# ==================================================================================================

print()
print("=" * 110)
print("CISLR — L31 + L32 UPDATED")
print("=" * 110)

print()
print("Recognition:")
print("  L26C model = sign identity")
print("  L27        = NO_SIGN rejection only")
print()

print("UPDATED sequence logic:")
print()
print("  FRIDAY -> FRIDAY")
print("      one token only")
print()
print("  FRIDAY -> MOTHER")
print("      friday mother")
print()
print("  NO_SIGN is NOT required between different valid signs.")
print()
print("  3-of-5 model confirmation is still required.")
print()

print("L32:")
print("  Select expected sign.")
print("  Perform the sign.")
print("  Wait until CURRENT INTERPRETATION is correct/stable.")
print("  Click Evaluate Current.")
print()

print("Locked test remains untouched.")
print()

print("=" * 110)


app_root = tk.Tk()

app = CISLRApplication(
    app_root
)

app_root.mainloop()

[PASS] L30 recognition pipeline detected.

CISLR — L31 + L32 UPDATED

Recognition:
  L26C model = sign identity
  L27        = NO_SIGN rejection only

UPDATED sequence logic:

  FRIDAY -> FRIDAY
      one token only

  FRIDAY -> MOTHER
      friday mother

  NO_SIGN is NOT required between different valid signs.

  3-of-5 model confirmation is still required.

L32:
  Select expected sign.
  Perform the sign.
  Wait until CURRENT INTERPRETATION is correct/stable.
  Click Evaluate Current.

Locked test remains untouched.

[SIGN CHANGE] INTEREST -> RUBBER
[SIGN CHANGE] RUBBER -> INTEREST
[SIGN CHANGE] INTEREST -> FRIDAY
[SIGN CHANGE] INTEREST -> THURSDAY
[SIGN CHANGE] THURSDAY -> MONDAY
[SIGN CHANGE] MONDAY -> CHRISTMAS
[SIGN CHANGE] INTEREST -> MONDAY
[SIGN CHANGE] MONDAY -> MOTHER
[SIGN CHANGE] FRIDAY -> PENCIL
[SIGN CHANGE] PENCIL -> FRIDAY
[SIGN CHANGE] FRIDAY -> CHRISTMAS
[SIGN CHANGE] MONDAY -> INTEREST
[SIGN CHANGE] INTEREST -> MONDAY
[SIGN CHANGE] MONDAY -> FRIDAY
[SIGN CHANGE] FR